## Fambig Analysis

In [ ]:
# =====================================================================
# CONFIG — change this one variable per run
# =====================================================================
# Options:
#   "fambig"        ->  tests  _fambig_full.root
#   "momerr"        ->  tests  _momerr_x2.root
#   "both"          ->  tests  _fambigFull_momerrX2.root
#   "momerr_pm50"   ->  tests both  _momerr_plus50pct.root  AND  _momerr_minus50pct.root

SCENARIO = "fambig"
# =====================================================================

import numpy as np
from sklearn.metrics import roc_curve, auc

# ---------------------------------------------------------------------
# Map scenario -> file(s)
# ---------------------------------------------------------------------
def files_for(scenario):
    base = training_dataset_filename
    if scenario == "fambig":
        return [("fambig = 1.0",  base.replace('.root', '_fambig_full.root'))]
    if scenario == "momerr":
        return [("momerr × 2",    base.replace('.root', '_momerr_x2.root'))]
    if scenario == "both":
        return [("Both stressed", base.replace('.root', '_fambigFull_momerrX2.root'))]
    if scenario == "momerr_pm50":
        return [("momerr +50%", base.replace('.root', '_momerr_plus50pct.root')),
                ("momerr -50%", base.replace('.root', '_momerr_minus50pct.root'))]
    raise ValueError(f"Unknown scenario: {scenario}")

# ---------------------------------------------------------------------
# Header
# ---------------------------------------------------------------------
print(f"{'Dataset':<22} {'AUC':>8}  {'Sig eff':>10}  {'BDT cut':>9}  "
      f"{'Mean shift':>12}  {'% sig flipped':>14}  {'Δsig':>7}  {'Δbkg':>7}")
print("-" * 110)

# ---------------------------------------------------------------------
# Baseline numbers (computed once)
# ---------------------------------------------------------------------
fpr0, tpr0, thr0 = roc_curve(y_full, y_pred_original_full, pos_label=1)
idx0    = np.where(1 - fpr0 >= 0.99)[0]
cut_idx = idx0[-1] if len(idx0) > 0 else 0
bdt_cut = thr0[cut_idx]
print(f"{'Baseline (orig)':<22} {auc(fpr0,tpr0):>8.4f}  "
      f"{tpr0[cut_idx]*100:>9.2f}%  {bdt_cut:>9.4f}  "
      f"{0.0:>12.6f}  {0.0:>13.2f}%  {0:>7}  {0:>7}")

# ---------------------------------------------------------------------
# Loop over the file(s) for this scenario
# ---------------------------------------------------------------------
for label, fname in files_for(SCENARIO):
    # 1) Get features for this stressed file
    x_new = get_features_from_file(fname)

    # 2) Run original model — no retraining
    y_new = model.predict_proba(x_new)[:, 1]

    # 3) Per-track prediction shift
    delta = y_new - y_pred_original_full

    # 4) ROC + AUC
    fpr, tpr, thr = roc_curve(y_full, y_new, pos_label=1)
    auc_score     = auc(fpr, tpr)

    # 5) Optimal cut at 99% BG rejection
    idx        = np.where(1 - fpr >= 0.99)[0]
    cut_i      = idx[-1] if len(idx) > 0 else 0
    sig_eff    = tpr[cut_i]
    opt_cut    = thr[cut_i]

    # 6) Track flipping at the ORIGINAL cut (bdt_cut)
    pass_orig  = y_pred_original_full >= bdt_cut
    pass_new   = y_new                >= bdt_cut
    flipped    = (pass_orig != pass_new)
    sig_flipped = (flipped & (y_full == 1)).sum()
    bkg_flipped = (flipped & (y_full == 0)).sum()
    pct_sig     = sig_flipped / N_signal * 100.0

    print(f"{label:<22} {auc_score:>8.4f}  {sig_eff*100:>9.2f}%  "
          f"{opt_cut:>9.4f}  {delta.mean():>+12.6f}  "
          f"{pct_sig:>13.2f}%  {sig_flipped:>7}  {bkg_flipped:>7}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, auc

# =====================================================================
# CONFIG — change to "fambig", "momerr", "both", or "momerr_pm50"
# =====================================================================
SCENARIO = "fambig"
# =====================================================================

# ---------------------------------------------------------------------
# Map scenario -> list of (label, color, scores_array)
# (the scores arrays are produced further down — this is just metadata)
# ---------------------------------------------------------------------
def scenario_meta(s):
    if s == "fambig":
        return {
            "title":    "BDT Score: Original vs fambig = 1.0",
            "rows":     [("Original",      "steelblue", 2.0),
                         ("fambig = 1.0",  "orange",    1.5)],
            "ylim":     (0.85, 1.0),
        }
    if s == "momerr":
        return {
            "title":    "BDT Score: Original vs momerr × 2",
            "rows":     [("Original",      "steelblue", 2.0),
                         ("momerr × 2",    "orange",    1.5)],
            "ylim":     (0.85, 1.0),
        }
    if s == "both":
        return {
            "title":    "BDT Score: Original vs Both stressed (fambig=1 & momerr×2)",
            "rows":     [("Original",       "steelblue", 2.0),
                         ("Both stressed",  "orange",    1.5)],
            "ylim":     (0.85, 1.0),
        }
    if s == "momerr_pm50":
        return {
            "title":    "BDT Score: Original vs momerr Heavy Perturbations (±50%)",
            "rows":     [("Original",      "steelblue", 2.0),
                         ("momerr +50%",   "orange",    1.5),
                         ("momerr -50%",   "darkblue",  1.5)],
            "ylim":     (0.85, 1.0),
        }
    raise ValueError(s)

meta = scenario_meta(SCENARIO)

# ---------------------------------------------------------------------
# Resolve filenames for this scenario
# ---------------------------------------------------------------------
def stress_filename(scenario, variant):
    base = training_dataset_filename
    return {
        "fambig_full"      : base.replace('.root', '_fambig_full.root'),
        "momerr_x2"        : base.replace('.root', '_momerr_x2.root'),
        "both"             : base.replace('.root', '_fambigFull_momerrX2.root'),
        "momerr_plus50"    : base.replace('.root', '_momerr_plus50pct.root'),
        "momerr_minus50"   : base.replace('.root', '_momerr_minus50pct.root'),
    }[variant]

# ---------------------------------------------------------------------
# Build the perturbed score arrays for this scenario
# ---------------------------------------------------------------------
def scores_for(scenario):
    if scenario == "fambig":
        x = get_features_from_file(stress_filename(scenario, "fambig_full"))
        return model.predict_proba(x)[:, 1]
    if scenario == "momerr":
        x = get_features_from_file(stress_filename(scenario, "momerr_x2"))
        return model.predict_proba(x)[:, 1]
    if scenario == "both":
        x = get_features_from_file(stress_filename(scenario, "both"))
        return model.predict_proba(x)[:, 1]
    if scenario == "momerr_pm50":
        x_up   = get_features_from_file(stress_filename(scenario, "momerr_plus50"))
        x_down = get_features_from_file(stress_filename(scenario, "momerr_minus50"))
        return model.predict_proba(x_up)[:, 1], model.predict_proba(x_down)[:, 1]

if SCENARIO == "momerr_pm50":
    y_new_up, y_new_down = scores_for(SCENARIO)
    perturbed = {
        "momerr +50%" : y_new_up,
        "momerr -50%" : y_new_down,
    }
else:
    y_new = scores_for(SCENARIO)
    perturbed = {meta["rows"][1][0]: y_new}

# ---------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
fig.suptitle(meta["title"], fontsize=16, fontweight='bold')

# Build the dataset list (always include Original at front)
datasets = [(y_pred_original_full, "Original", "steelblue", 2.0)]
for label, color, lw in meta["rows"][1:]:
    datasets.append((perturbed[label], label, color, lw))

for ax, panel_title, xlim in zip(
        axes,
        ["Full range (log scale)", "Zoomed around cut"],
        [None, meta["ylim"]]):

    for scores, label, color, lw in datasets:
        ax.hist(scores, bins=100, histtype="step",
                linewidth=lw, color=color, label=label)

    # ----- BASELINE cut (black dashed) -----
    ax.axvline(trkqual_cut, color='black', linestyle='--', linewidth=1.5,
               label=f'Baseline cut = {trkqual_cut:.4f}')

    # ----- CORRECTED cut per dataset at 99% BG rejection -----
    # solid thin vertical line for each perturbed file
    palette = {"Original": "steelblue"}
    for i, (label, color, _) in enumerate(meta["rows"][1:]):
        scores = perturbed[label]
        fpr, tpr, thr = roc_curve(y_full, scores, pos_label=1)
        idx = np.where(1 - fpr >= 0.99)[0]
        if len(idx) > 0:
            ci   = idx[-1]
            cut  = thr[ci]
            ax.axvline(cut, color=color, linestyle=':', linewidth=1.5,
                       label=f'{label} cut (99% BG rej) = {cut:.4f}')

    ax.set_xlabel("BDT Score")
    ax.set_ylabel("Tracks")
    ax.set_title(panel_title)
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(alpha=0.3)
    if xlim:
        ax.set_xlim(xlim)
    else:
        ax.set_yscale('log')

# ---------------------------------------------------------------------
# Build summary table
# ---------------------------------------------------------------------
N_signal = (y_full == 1).sum()
bdt_pass_orig = y_pred_original_full >= trkqual_cut

table_data  = [["Original Baseline", "—", "—", "—", "—", "—"]]
columns     = ["Configuration", "Mean Shift", "Changed", "% of Signal", "Signal", "Bkg"]

for label, color, _ in meta["rows"][1:]:
    y_new       = perturbed[label]
    delta       = y_new - y_pred_original_full
    bdt_pass    = y_new >= trkqual_cut
    changed     = (bdt_pass_orig != bdt_pass).sum()
    sig_changed = ((bdt_pass_orig != bdt_pass) & (y_full == 1)).sum()
    bkg_changed = ((bdt_pass_orig != bdt_pass) & (y_full == 0)).sum()
    table_data.append([
        label,
        f"{delta.mean():+.6f}",
        f"{changed:,}",
        f"{sig_changed/N_signal*100:.2f}%",
        f"{sig_changed:,}",
        f"{bkg_changed:,}",
    ])

# ---------------------------------------------------------------------
# Render the table below the plots
# ---------------------------------------------------------------------
n_rows = len(table_data) + 1
table_ax = fig.add_axes([0.05, -0.04 - 0.05*n_rows, 0.9, 0.05*n_rows])
table_ax.axis('off')

summary_table = table_ax.table(cellText=table_data, colLabels=columns,
                               loc='center', cellLoc='center')
summary_table.auto_set_font_size(False)
summary_table.set_fontsize(12)
summary_table.scale(1.0, 1.6)

for (row, col), cell in summary_table.get_celld().items():
    if col == 0 and row > 0:
        cell.set_text_props(ha='left')
    if row == 0:
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#2f4f4f')

plt.subplots_adjust(bottom=0.18 + 0.05*n_rows)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, auc
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

# =====================================================================
# CONFIG — change this to switch scenarios
# =====================================================================
# Options:
#   "fambig"   ->  just fambig = 1.0
#   "momerr"   ->  just momerr × 2
#   "both"     ->  combined fambig=1 + momerr×2
#   "all"      ->  momerr ±10%, ±50%, fambig=1, combined, original
# =====================================================================
SCENARIO = "fambig"
# =====================================================================

# ---------------------------------------------------------------------
# Map scenario -> list of (filename, label, color)
# ---------------------------------------------------------------------
def datasets_for(scenario):
    base = training_dataset_filename

    common = [
        (base, "Original", "steelblue"),
    ]
    perturbed = {
        "fambig" : [(base.replace('.root', '_fambig_full.root'),
                     "fambig = 1.0", "orange")],
        "momerr" : [(base.replace('.root', '_momerr_x2.root'),
                     "momerr × 2", "orange")],
        "both"   : [(base.replace('.root', '_fambigFull_momerrX2.root'),
                     "Both stressed", "orange")],
        "all"    : [
            (base.replace('.root', '_momerr_plus10pct.root'),
             "momerr +10%", "tomato"),
            (base.replace('.root', '_momerr_minus10pct.root'),
             "momerr -10%", "limegreen"),
            (base.replace('.root', '_momerr_plus50pct.root'),
             "momerr +50%", "orange"),
            (base.replace('.root', '_momerr_minus50pct.root'),
             "momerr -50%", "darkblue"),
            (base.replace('.root', '_fambig_full.root'),
             "fambig = 1.0", "purple"),
            (base.replace('.root', '_fambigFull_momerrX2.root'),
             "Both stressed", "crimson"),
        ],
    }
    return common + perturbed[scenario]

dataset_meta = datasets_for(SCENARIO)

# ---------------------------------------------------------------------
# Load / compute (score, label) for each entry
# ---------------------------------------------------------------------
def load_scores(fname):
    x = get_features_from_file(fname)
    return model.predict_proba(x)[:, 1]

datasets_roc = []
for fname, label, color in dataset_meta:
    scores = (y_pred_original_full if fname == training_dataset_filename
              else load_scores(fname))
    datasets_roc.append((scores, y_full, label, color))

# ---------------------------------------------------------------------
# Plot parameters
# ---------------------------------------------------------------------
static_cut_target = 0.92
rej_target        = 0.99

# ---------------------------------------------------------------------
# Initialize side-by-side subplots
# ---------------------------------------------------------------------
n_panels = len(datasets_roc)
fig_w = max(22, 14 + 1.5 * (n_panels - 4))   # wider if many panels
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(fig_w, 10))
fig.suptitle(f"ROC Curve Analysis — Static vs Adaptive Cut  "
             f"(scenario: {SCENARIO})",
             fontsize=22, fontweight='bold', y=0.96)

ax_ins1 = inset_axes(ax1, width="40%", height="40%", loc='upper right',
                     bbox_to_anchor=(-0.02, -0.02, 1, 1),
                     bbox_transform=ax1.transAxes)
ax_ins2 = inset_axes(ax2, width="40%", height="40%", loc='upper right',
                     bbox_to_anchor=(-0.02, -0.02, 1, 1),
                     bbox_transform=ax2.transAxes)

# ---------------------------------------------------------------------
# Loop over datasets
# ---------------------------------------------------------------------
for scores, labels, name, color in datasets_roc:
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    auc_score = auc(fpr, tpr)

    # ---- LEFT PANEL: STATIC CUT ----
    idx_static = np.where(thresholds >= static_cut_target)[0]
    idx_s = idx_static[-1] if len(idx_static) > 0 else 0
    sig_eff_s = float(tpr[idx_s])
    bkg_rej_s = float(1 - fpr[idx_s])

    ax1.plot(tpr, 1 - fpr, color=color, linewidth=2,
             label=f"{name:<14} | AUC={auc_score:.4f} | "
                   f"Eff={sig_eff_s*100:5.1f}% | "
                   f"Rej={bkg_rej_s*100:5.2f}%")
    ax1.plot(tpr[idx_s], 1 - fpr[idx_s], 'o', color=color, markersize=8)
    ax_ins1.plot(tpr, 1 - fpr, color=color, linewidth=1.5)
    ax_ins1.plot(tpr[idx_s], 1 - fpr[idx_s], 'o', color=color, markersize=6)

    # ---- RIGHT PANEL: ADAPTIVE CUT ----
    idx_dyn = np.where(1 - fpr >= rej_target)[0]
    idx_d = idx_dyn[-1] if len(idx_dyn) > 0 else 0
    sig_eff_d = float(tpr[idx_d])
    opt_cut   = float(thresholds[idx_d])

    ax2.plot(tpr, 1 - fpr, color=color, linewidth=2,
             label=f"{name:<14} | AUC={auc_score:.4f} | "
                   f"Cut={opt_cut:.3f} | "
                   f"Eff={sig_eff_d*100:5.1f}%")
    ax2.plot(tpr[idx_d], 1 - fpr[idx_d], 'o', color=color, markersize=8)
    ax_ins2.plot(tpr, 1 - fpr, color=color, linewidth=1.5)
    ax_ins2.plot(tpr[idx_d], 1 - fpr[idx_d], 'o', color=color, markersize=6)

# ---------------------------------------------------------------------
# Format LEFT subplot
# ---------------------------------------------------------------------
ax1.plot([0, 1], [1, 0], 'k--', alpha=0.3)
ax1.set_xlabel("Signal Efficiency (TPR)", fontsize=12, fontweight='bold')
ax1.set_ylabel("Background Rejection (1 − FPR)", fontsize=12, fontweight='bold')
ax1.set_title(f"A. Operating Points at Static BDT Cut = {static_cut_target:.2f}",
              fontsize=13, fontweight='bold', color='crimson')
ax1.grid(alpha=0.3)
ax1.legend(loc='lower left', fontsize=11, framealpha=0.9)

ax_ins1.set_xlim(0.35, 0.82)
ax_ins1.set_ylim(0.915, 1.002)
ax_ins1.set_title("Operating-point scatter (zoom)", fontsize=9, fontweight='bold')
ax_ins1.grid(alpha=0.3)
mark_inset(ax1, ax_ins1, loc1=2, loc2=3, fc="none", ec="0.5", ls="--")

# ---------------------------------------------------------------------
# Format RIGHT subplot
# ---------------------------------------------------------------------
ax2.plot([0, 1], [1, 0], 'k--', alpha=0.3)
ax2.set_xlabel("Signal Efficiency (TPR)", fontsize=12, fontweight='bold')
ax2.set_ylabel("Background Rejection (1 − FPR)", fontsize=12, fontweight='bold')
ax2.set_title(f"B. Operating Points at Adaptive Cut (Rej = {rej_target*100:.0f}%)",
              fontsize=13, fontweight='bold', color='darkgreen')
ax2.grid(alpha=0.3)
ax2.legend(loc='lower left', fontsize=11, framealpha=0.9)

ax_ins2.set_xlim(0.50, 0.62)
ax_ins2.set_ylim(0.988, 1.001)
ax_ins2.set_title("Stabilized 99% rejection line (zoom)", fontsize=9, fontweight='bold')
ax_ins2.grid(alpha=0.3)
mark_inset(ax2, ax_ins2, loc1=2, loc2=3, fc="none", ec="0.5", ls="--")

plt.tight_layout(rect=[0, 0, 1, 0.94])
# plt.savefig(f"roc_robustness_{SCENARIO}.png", dpi=200, bbox_inches='tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =====================================================================
# CONFIG — pick the scenario to overlay
# =====================================================================
# Options:
#   "fambig"  -> overlay  _fambig_full.root
#   "momerr"  -> overlay  _momerr_x2.root
#   "both"    -> overlay  _fambigFull_momerrX2.root
# =====================================================================
SCENARIO = "fambig"
# =====================================================================

# ---------------------------------------------------------------------
# Resolve file(s) and color/style for the scenario
# ---------------------------------------------------------------------
def perturbed_files(scenario):
    base = training_dataset_filename
    if scenario == "fambig":
        return [(base.replace('.root', '_fambig_full.root'),
                 "fambig = 1.0", "orange", "--")]
    if scenario == "momerr":
        return [(base.replace('.root', '_momerr_x2.root'),
                 "momerr × 2", "orange", "--")]
    if scenario == "both":
        return [(base.replace('.root', '_fambigFull_momerrX2.root'),
                 "Both stressed", "orange", "--")]
    raise ValueError(scenario)

perturbed = perturbed_files(SCENARIO)

# ---------------------------------------------------------------------
# Load feature arrays for the perturbed file
# ---------------------------------------------------------------------
x_perturbed = {}
for fname, label, color, ls in perturbed:
    x_perturbed[label] = get_features_from_file(fname)

# ---------------------------------------------------------------------
# Figure setup
# ---------------------------------------------------------------------
n_input_vars = len(input_var_names)
n_rows, n_cols = 3, 3
fig, axs = plt.subplots(n_rows, n_cols, figsize=(16, 9))
fig.subplots_adjust(hspace=0.5, wspace=0.3)

title_str = "Feature Distributions — Full Dataset vs " + SCENARIO.upper()
fig.suptitle(title_str, fontsize=13, fontweight='bold')

# ---------------------------------------------------------------------
# Loop over all input variables
# ---------------------------------------------------------------------
for i_var in range(n_input_vars):
    ax = axs.flatten()[i_var]

    # Baseline distribution
    ax.hist(x_full[:, i_var], bins=100,
            range=(x_mins[i_var], x_maxs[i_var]),
            histtype='step', color='black',
            log=log_ys[i_var],
            label=f'Full dataset (N={len(x_full):,})',
            linewidth=1.5)

    # Overlay the perturbed file
    for fname, label, color, ls in perturbed:
        x_arr = x_perturbed[label]
        # Skip if the feature is unchanged in this perturbation
        if np.array_equal(x_arr[:, i_var], x_full[:, i_var]):
            continue
        ax.hist(x_arr[:, i_var], bins=100,
                range=(x_mins[i_var], x_maxs[i_var]),
                histtype='step', color=color, ls=ls,
                log=log_ys[i_var],
                label=label, linewidth=1.4)

    ax.set_xlabel(input_var_names[i_var] + " " + units[i_var])
    ax.margins(0)
    ax.legend(fontsize=7, loc='best')

# Hide unused axes if n_input_vars < 9
for j in range(n_input_vars, n_rows * n_cols):
    axs.flatten()[j].axis('off')

# plt.savefig(f"feature_distributions_{SCENARIO}.png", dpi=150, bbox_inches='tight")
plt.show()


In [ ]:
i = input_var_names.index("fambig")

x_full = get_features_from_file(training_dataset_filename)
x_amb  = get_features_from_file(
    training_dataset_filename.replace('.root', '_fambig_full.root')
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

plot_params_fambig = [
    (x_full[:, i], 'black',  f'Full dataset (N={len(x_full):,})', '-',  2.0),
    (x_amb[:,  i], 'orange', 'fambig = 1.0',                     '--', 1.5),
]

for ax in [ax1, ax2]:
    for data, color, label, style, lw in plot_params_fambig:
        ax.hist(data, bins=100,
                range=(x_mins[i], x_maxs[i]) if ax == ax1 else (0, 1.0),
                histtype='step', color=color, log=True,
                label=label, linestyle=style, linewidth=lw)
    ax.set_ylabel("Tracks")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

ax1.set_title("Full Range Distribution", fontweight='bold')
ax1.set_xlabel(f"{input_var_names[i]} {units[i]}")
ax1.margins(0)
ax2.set_title("Zoomed Distribution (0–1)", fontweight='bold')
ax2.set_xlabel("fambig [unitless]")
fig.suptitle("fambig Distribution — Forced Ambiguity vs Original",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
import uproot
import numpy as np
import matplotlib.pyplot as plt

# =====================================================================
# CONFIG
# =====================================================================
SCENARIO    = "fambig"        # "fambig" | "momerr" | "both"
static_cut  = 0.92            # Original's correct cut (used in BOTH panels)
correct_cut = 0.8005          # Stressed file's correct cut (right panel only)
# =====================================================================

# ---------------------------------------------------------------------
# Resolve perturbed file
# ---------------------------------------------------------------------
def stressed_file(scenario):
    base = training_dataset_filename
    return {
        "fambig": (base.replace('.root', '_fambig_full.root'),
                   "fambig = 1.0", "orange", "--"),
        "momerr": (base.replace('.root', '_momerr_x2.root'),
                   "momerr × 2",   "orange", "--"),
        "both"  : (base.replace('.root', '_fambigFull_momerrX2.root'),
                   "Both stressed", "crimson", "-."),
    }[scenario]

f_stressed, label_stressed, color_stressed, ls_stressed = stressed_file(SCENARIO)

# ---------------------------------------------------------------------
# 1. Load BASELINE arrays
# ---------------------------------------------------------------------
arrays = uproot.open(training_dataset_filename)[training_dataset_treename].arrays(library="np")

nactive     = arrays['trk.nactive']
nhits       = arrays['trk.nhits']
factive     = nactive / nhits
fambig      = arrays['trk.nnullambig'] / nactive
fstraws     = arrays['trk.nmatactive'] / nactive
t0err       = arrays['trk_ent_pars.t0err']
fitcon      = arrays['trk.fitcon']
momerr_raw  = arrays['trk_ent.momerr']

p_reco = (arrays['trk_ent.mom.fCoordinates.fX']**2 +
          arrays['trk_ent.mom.fCoordinates.fY']**2 +
          arrays['trk_ent.mom.fCoordinates.fZ']**2)**0.5
p_true = (arrays['trk_ent_mc.mom.fCoordinates.fX']**2 +
          arrays['trk_ent_mc.mom.fCoordinates.fY']**2 +
          arrays['trk_ent_mc.mom.fCoordinates.fZ']**2)**0.5
mom_res = p_reco - p_true

x_matrix = np.vstack((nactive, factive, t0err, fambig, fitcon,
                      momerr_raw, fstraws)).T
scores_orig = model.predict_proba(x_matrix)[:, 1]

# ---------------------------------------------------------------------
# 2. Load STRESSED arrays
# ---------------------------------------------------------------------
arrays_s = uproot.open(f_stressed)[training_dataset_treename].arrays(library="np")

nactive_s     = arrays_s['trk.nactive']
factive_s     = nactive_s / arrays_s['trk.nhits']
fambig_s      = arrays_s['trk.nnullambig'] / nactive_s
fstraws_s     = arrays_s['trk.nmatactive'] / nactive_s
t0err_s       = arrays_s['trk_ent_pars.t0err']
fitcon_s      = arrays_s['trk.fitcon']
momerr_s      = arrays_s['trk_ent.momerr']

x_stressed = np.vstack((nactive_s, factive_s, t0err_s, fambig_s,
                        fitcon_s, momerr_s, fstraws_s)).T
scores_stressed = model.predict_proba(x_stressed)[:, 1]

# ---------------------------------------------------------------------
# 3. Plot
# ---------------------------------------------------------------------
n_bins = 100
mom_min, mom_max = -10.0, 10.0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle(f"Momentum Resolution — Original @ 0.92 (both panels)  vs  "
             f"{label_stressed} @ 0.92 (left) and @{correct_cut} (right)",
             fontsize=18, fontweight='bold')

# Background (all tracks) on both panels
for ax in [ax1, ax2]:
    ax.hist(mom_res, bins=n_bins, range=(mom_min, mom_max),
            log=True, histtype='step', color='black',
            label='All tracks (unfiltered)', linewidth=2.0, alpha=0.7)

# Original @ static_cut = 0.92 — IDENTICAL in both panels
for ax in [ax1, ax2]:
    ax.hist(mom_res[scores_orig >= static_cut],
            bins=n_bins, range=(mom_min, mom_max),
            log=True, histtype='step', color='steelblue',
            linewidth=1.8,
            label=f'Original (Cut = {static_cut})')

# Left panel — Stressed @ SAME cut as Original (= 0.92)
ax1.hist(mom_res[scores_stressed >= static_cut],
         bins=n_bins, range=(mom_min, mom_max),
         log=True, histtype='step', color=color_stressed, linestyle=ls_stressed,
         linewidth=1.8,
         label=f'{label_stressed} (Cut = {static_cut})')

# Right panel — Stressed @ its OWN correct cut
ax2.hist(mom_res[scores_stressed >= correct_cut],
         bins=n_bins, range=(mom_min, mom_max),
         log=True, histtype='step', color=color_stressed, linestyle=ls_stressed,
         linewidth=1.8,
         label=f'{label_stressed} (Cut = {correct_cut})')

# Format
ax1.set_xlim(mom_min, mom_max)
ax1.set_xlabel("Momentum Resolution [MeV/c]", fontweight='bold', fontsize=11)
ax1.set_ylabel("Number of Tracks (Log Scale)", fontweight='bold', fontsize=11)
ax1.set_title(f"A. Both files at Static Cut = {static_cut}",
              fontsize=13, fontweight='bold', color='crimson')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='upper right', fontsize=11, framealpha=0.9)

ax2.set_xlim(mom_min, mom_max)
ax2.set_xlabel("Momentum Resolution [MeV/c]", fontweight='bold', fontsize=11)
ax2.set_ylabel("Number of Tracks (Log Scale)", fontweight='bold', fontsize=11)
ax2.set_title(f"B. Original @ {static_cut} vs {label_stressed} @ {correct_cut}",
              fontsize=13, fontweight='bold', color='darkgreen')
ax2.grid(True, alpha=0.3)
ax2.legend(loc='upper right', fontsize=11, framealpha=0.9)

plt.tight_layout()
# plt.savefig(f"mom_resolution_{SCENARIO}.png", dpi=200, bbox_inches='tight')
plt.show()


## Momerr %100

In [ ]:
# =====================================================================
# CONFIG — change this one variable per run
# =====================================================================
# Options:
#   "fambig"        ->  tests  _fambig_full.root
#   "momerr"        ->  tests  _momerr_x2.root
#   "both"          ->  tests  _fambigFull_momerrX2.root
#   "momerr_pm50"   ->  tests both  _momerr_plus50pct.root  AND  _momerr_minus50pct.root

SCENARIO = "momerr" 
# =====================================================================

import numpy as np
from sklearn.metrics import roc_curve, auc

# ---------------------------------------------------------------------
# Map scenario -> file(s)
# ---------------------------------------------------------------------
def files_for(scenario):
    base = training_dataset_filename
    if scenario == "fambig":
        return [("fambig = 1.0",  base.replace('.root', '_fambig_full.root'))]
    if scenario == "momerr":
        return [("momerr × 2",    base.replace('.root', '_momerr_x2.root'))]
    if scenario == "both":
        return [("Both stressed", base.replace('.root', '_fambigFull_momerrX2.root'))]
    if scenario == "momerr_pm50":
        return [("momerr +50%", base.replace('.root', '_momerr_plus50pct.root')),
                ("momerr -50%", base.replace('.root', '_momerr_minus50pct.root'))]
    raise ValueError(f"Unknown scenario: {scenario}")

# ---------------------------------------------------------------------
# Header
# ---------------------------------------------------------------------
print(f"{'Dataset':<22} {'AUC':>8}  {'Sig eff':>10}  {'BDT cut':>9}  "
      f"{'Mean shift':>12}  {'% sig flipped':>14}  {'Δsig':>7}  {'Δbkg':>7}")
print("-" * 110)

# ---------------------------------------------------------------------
# Baseline numbers (computed once)
# ---------------------------------------------------------------------
fpr0, tpr0, thr0 = roc_curve(y_full, y_pred_original_full, pos_label=1)
idx0    = np.where(1 - fpr0 >= 0.99)[0]
cut_idx = idx0[-1] if len(idx0) > 0 else 0
bdt_cut = thr0[cut_idx]
print(f"{'Baseline (orig)':<22} {auc(fpr0,tpr0):>8.4f}  "
      f"{tpr0[cut_idx]*100:>9.2f}%  {bdt_cut:>9.4f}  "
      f"{0.0:>12.6f}  {0.0:>13.2f}%  {0:>7}  {0:>7}")

# ---------------------------------------------------------------------
# Loop over the file(s) for this scenario
# ---------------------------------------------------------------------
for label, fname in files_for(SCENARIO):
    # 1) Get features for this stressed file
    x_new = get_features_from_file(fname)

    # 2) Run original model — no retraining
    y_new = model.predict_proba(x_new)[:, 1]

    # 3) Per-track prediction shift
    delta = y_new - y_pred_original_full

    # 4) ROC + AUC
    fpr, tpr, thr = roc_curve(y_full, y_new, pos_label=1)
    auc_score     = auc(fpr, tpr)

    # 5) Optimal cut at 99% BG rejection
    idx        = np.where(1 - fpr >= 0.99)[0]
    cut_i      = idx[-1] if len(idx) > 0 else 0
    sig_eff    = tpr[cut_i]
    opt_cut    = thr[cut_i]

    # 6) Track flipping at the ORIGINAL cut (bdt_cut)
    pass_orig  = y_pred_original_full >= bdt_cut
    pass_new   = y_new                >= bdt_cut
    flipped    = (pass_orig != pass_new)
    sig_flipped = (flipped & (y_full == 1)).sum()
    bkg_flipped = (flipped & (y_full == 0)).sum()
    pct_sig     = sig_flipped / N_signal * 100.0

    print(f"{label:<22} {auc_score:>8.4f}  {sig_eff*100:>9.2f}%  "
          f"{opt_cut:>9.4f}  {delta.mean():>+12.6f}  "
          f"{pct_sig:>13.2f}%  {sig_flipped:>7}  {bkg_flipped:>7}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, auc

# CONFIG — change to "fambig", "momerr", "both", or "momerr_pm50"

SCENARIO = "momerr"

# Map scenario -> list of (label, color, scores_array)
# (the scores arrays are produced further down — this is just metadata)

def scenario_meta(s):
    if s == "fambig":
        return {
            "title":    "BDT Score: Original vs fambig = 1.0",
            "rows":     [("Original",      "steelblue", 2.0),
                         ("fambig = 1.0",  "orange",    1.5)],
            "ylim":     (0.85, 1.0),
        }
    if s == "momerr":
        return {
            "title":    "BDT Score: Original vs momerr × 2",
            "rows":     [("Original",      "steelblue", 2.0),
                         ("momerr × 2",    "orange",    1.5)],
            "ylim":     (0.85, 1.0),
        }
    if s == "both":
        return {
            "title":    "BDT Score: Original vs Both stressed (fambig=1 & momerr×2)",
            "rows":     [("Original",       "steelblue", 2.0),
                         ("Both stressed",  "orange",    1.5)],
            "ylim":     (0.85, 1.0),
        }
    if s == "momerr_pm50":
        return {
            "title":    "BDT Score: Original vs momerr Heavy Perturbations (±50%)",
            "rows":     [("Original",      "steelblue", 2.0),
                         ("momerr +50%",   "orange",    1.5),
                         ("momerr -50%",   "darkblue",  1.5)],
            "ylim":     (0.85, 1.0),
        }
    raise ValueError(s)

meta = scenario_meta(SCENARIO)

# Resolve filenames for this scenario

def stress_filename(scenario, variant):
    base = training_dataset_filename
    return {
        "fambig_full"      : base.replace('.root', '_fambig_full.root'),
        "momerr_x2"        : base.replace('.root', '_momerr_x2.root'),
        "both"             : base.replace('.root', '_fambigFull_momerrX2.root'),
        "momerr_plus50"    : base.replace('.root', '_momerr_plus50pct.root'),
        "momerr_minus50"   : base.replace('.root', '_momerr_minus50pct.root'),
    }[variant]

# Build the perturbed score arrays for this scenario

def scores_for(scenario):
    if scenario == "fambig":
        x = get_features_from_file(stress_filename(scenario, "fambig_full"))
        return model.predict_proba(x)[:, 1]
    if scenario == "momerr":
        x = get_features_from_file(stress_filename(scenario, "momerr_x2"))
        return model.predict_proba(x)[:, 1]
    if scenario == "both":
        x = get_features_from_file(stress_filename(scenario, "both"))
        return model.predict_proba(x)[:, 1]
    if scenario == "momerr_pm50":
        x_up   = get_features_from_file(stress_filename(scenario, "momerr_plus50"))
        x_down = get_features_from_file(stress_filename(scenario, "momerr_minus50"))
        return model.predict_proba(x_up)[:, 1], model.predict_proba(x_down)[:, 1]

if SCENARIO == "momerr_pm50":
    y_new_up, y_new_down = scores_for(SCENARIO)
    perturbed = {
        "momerr +50%" : y_new_up,
        "momerr -50%" : y_new_down,
    }
else:
    y_new = scores_for(SCENARIO)
    perturbed = {meta["rows"][1][0]: y_new}

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
fig.suptitle(meta["title"], fontsize=16, fontweight='bold')

# Build the dataset list (always include Original at front)
datasets = [(y_pred_original_full, "Original", "steelblue", 2.0)]
for label, color, lw in meta["rows"][1:]:
    datasets.append((perturbed[label], label, color, lw))

for ax, panel_title, xlim in zip(
        axes,
        ["Full range (log scale)", "Zoomed around cut"],
        [None, meta["ylim"]]):

    for scores, label, color, lw in datasets:
        ax.hist(scores, bins=100, histtype="step",
                linewidth=lw, color=color, label=label)

    # BASELINE cut (black dashed)
    ax.axvline(trkqual_cut, color='black', linestyle='--', linewidth=1.5,
               label=f'Baseline cut = {trkqual_cut:.4f}')

    # CORRECTED cut per dataset at 99% BG rejection
    # solid thin vertical line for each perturbed file
    palette = {"Original": "steelblue"}
    for i, (label, color, _) in enumerate(meta["rows"][1:]):
        scores = perturbed[label]
        fpr, tpr, thr = roc_curve(y_full, scores, pos_label=1)
        idx = np.where(1 - fpr >= 0.99)[0]
        if len(idx) > 0:
            ci   = idx[-1]
            cut  = thr[ci]
            ax.axvline(cut, color=color, linestyle=':', linewidth=1.5,
                       label=f'{label} cut (99% BG rej) = {cut:.4f}')

    ax.set_xlabel("BDT Score")
    ax.set_ylabel("Tracks")
    ax.set_title(panel_title)
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(alpha=0.3)
    if xlim:
        ax.set_xlim(xlim)
    else:
        ax.set_yscale('log')

# Build summary table

N_signal = (y_full == 1).sum()
bdt_pass_orig = y_pred_original_full >= trkqual_cut

table_data  = [["Original Baseline", "—", "—", "—", "—", "—"]]
columns     = ["Configuration", "Mean Shift", "Changed", "% of Signal", "Signal", "Bkg"]

for label, color, _ in meta["rows"][1:]:
    y_new       = perturbed[label]
    delta       = y_new - y_pred_original_full
    bdt_pass    = y_new >= trkqual_cut
    changed     = (bdt_pass_orig != bdt_pass).sum()
    sig_changed = ((bdt_pass_orig != bdt_pass) & (y_full == 1)).sum()
    bkg_changed = ((bdt_pass_orig != bdt_pass) & (y_full == 0)).sum()
    table_data.append([
        label,
        f"{delta.mean():+.6f}",
        f"{changed:,}",
        f"{sig_changed/N_signal*100:.2f}%",
        f"{sig_changed:,}",
        f"{bkg_changed:,}",
    ])

# Render the table below the plots

n_rows = len(table_data) + 1
table_ax = fig.add_axes([0.05, -0.04 - 0.05*n_rows, 0.9, 0.05*n_rows])
table_ax.axis('off')

summary_table = table_ax.table(cellText=table_data, colLabels=columns,
                               loc='center', cellLoc='center')
summary_table.auto_set_font_size(False)
summary_table.set_fontsize(12)
summary_table.scale(1.0, 1.6)

for (row, col), cell in summary_table.get_celld().items():
    if col == 0 and row > 0:
        cell.set_text_props(ha='left')
    if row == 0:
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#2f4f4f')

plt.subplots_adjust(bottom=0.18 + 0.05*n_rows)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, auc
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

# CONFIG — change this to switch scenarios

# Options:
#   "fambig"   ->  just fambig = 1.0
#   "momerr"   ->  just momerr × 2
#   "both"     ->  combined fambig=1 + momerr×2
#   "all"      ->  momerr ±10%, ±50%, fambig=1, combined, original

SCENARIO = "momerr"

# Map scenario -> list of (filename, label, color)

def datasets_for(scenario):
    base = training_dataset_filename

    common = [
        (base, "Original", "steelblue"),
    ]
    perturbed = {
        "fambig" : [(base.replace('.root', '_fambig_full.root'),
                     "fambig = 1.0", "orange")],
        "momerr" : [(base.replace('.root', '_momerr_x2.root'),
                     "momerr × 2", "orange")],
        "both"   : [(base.replace('.root', '_fambigFull_momerrX2.root'),
                     "Both stressed", "orange")],
        "all"    : [
            (base.replace('.root', '_momerr_plus10pct.root'),
             "momerr +10%", "tomato"),
            (base.replace('.root', '_momerr_minus10pct.root'),
             "momerr -10%", "limegreen"),
            (base.replace('.root', '_momerr_plus50pct.root'),
             "momerr +50%", "orange"),
            (base.replace('.root', '_momerr_minus50pct.root'),
             "momerr -50%", "darkblue"),
            (base.replace('.root', '_fambig_full.root'),
             "fambig = 1.0", "purple"),
            (base.replace('.root', '_fambigFull_momerrX2.root'),
             "Both stressed", "crimson"),
        ],
    }
    return common + perturbed[scenario]

dataset_meta = datasets_for(SCENARIO)

# Load / compute (score, label) for each entry

def load_scores(fname):
    x = get_features_from_file(fname)
    return model.predict_proba(x)[:, 1]

datasets_roc = []
for fname, label, color in dataset_meta:
    scores = (y_pred_original_full if fname == training_dataset_filename
              else load_scores(fname))
    datasets_roc.append((scores, y_full, label, color))

# Plot parameters

static_cut_target = 0.92
rej_target        = 0.99

# Initialize side-by-side subplots

n_panels = len(datasets_roc)
fig_w = max(22, 14 + 1.5 * (n_panels - 4))   # wider if many panels
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(fig_w, 10))
fig.suptitle(f"ROC Curve Analysis — Static vs Adaptive Cut  "
             f"(scenario: {SCENARIO})",
             fontsize=22, fontweight='bold', y=0.96)

ax_ins1 = inset_axes(ax1, width="40%", height="40%", loc='upper right',
                     bbox_to_anchor=(-0.02, -0.02, 1, 1),
                     bbox_transform=ax1.transAxes)
ax_ins2 = inset_axes(ax2, width="40%", height="40%", loc='upper right',
                     bbox_to_anchor=(-0.02, -0.02, 1, 1),
                     bbox_transform=ax2.transAxes)

# Loop over datasets

for scores, labels, name, color in datasets_roc:
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    auc_score = auc(fpr, tpr)

    # ---- LEFT PANEL: STATIC CUT ----
    idx_static = np.where(thresholds >= static_cut_target)[0]
    idx_s = idx_static[-1] if len(idx_static) > 0 else 0
    sig_eff_s = float(tpr[idx_s])
    bkg_rej_s = float(1 - fpr[idx_s])

    ax1.plot(tpr, 1 - fpr, color=color, linewidth=2,
             label=f"{name:<14} | AUC={auc_score:.4f} | "
                   f"Eff={sig_eff_s*100:5.1f}% | "
                   f"Rej={bkg_rej_s*100:5.2f}%")
    ax1.plot(tpr[idx_s], 1 - fpr[idx_s], 'o', color=color, markersize=8)
    ax_ins1.plot(tpr, 1 - fpr, color=color, linewidth=1.5)
    ax_ins1.plot(tpr[idx_s], 1 - fpr[idx_s], 'o', color=color, markersize=6)

    # ---- RIGHT PANEL: ADAPTIVE CUT ----
    idx_dyn = np.where(1 - fpr >= rej_target)[0]
    idx_d = idx_dyn[-1] if len(idx_dyn) > 0 else 0
    sig_eff_d = float(tpr[idx_d])
    opt_cut   = float(thresholds[idx_d])

    ax2.plot(tpr, 1 - fpr, color=color, linewidth=2,
             label=f"{name:<14} | AUC={auc_score:.4f} | "
                   f"Cut={opt_cut:.3f} | "
                   f"Eff={sig_eff_d*100:5.1f}%")
    ax2.plot(tpr[idx_d], 1 - fpr[idx_d], 'o', color=color, markersize=8)
    ax_ins2.plot(tpr, 1 - fpr, color=color, linewidth=1.5)
    ax_ins2.plot(tpr[idx_d], 1 - fpr[idx_d], 'o', color=color, markersize=6)


# Format LEFT subplot

ax1.plot([0, 1], [1, 0], 'k--', alpha=0.3)
ax1.set_xlabel("Signal Efficiency (TPR)", fontsize=12, fontweight='bold')
ax1.set_ylabel("Background Rejection (1 − FPR)", fontsize=12, fontweight='bold')
ax1.set_title(f"A. Operating Points at Static BDT Cut = {static_cut_target:.2f}",
              fontsize=13, fontweight='bold', color='crimson')
ax1.grid(alpha=0.3)
ax1.legend(loc='lower left', fontsize=11, framealpha=0.9)

ax_ins1.set_xlim(0.35, 0.82)
ax_ins1.set_ylim(0.915, 1.002)
ax_ins1.set_title("Operating-point scatter (zoom)", fontsize=9, fontweight='bold')
ax_ins1.grid(alpha=0.3)
mark_inset(ax1, ax_ins1, loc1=2, loc2=3, fc="none", ec="0.5", ls="--")

# Format RIGHT subplot

ax2.plot([0, 1], [1, 0], 'k--', alpha=0.3)
ax2.set_xlabel("Signal Efficiency (TPR)", fontsize=12, fontweight='bold')
ax2.set_ylabel("Background Rejection (1 − FPR)", fontsize=12, fontweight='bold')
ax2.set_title(f"B. Operating Points at Adaptive Cut (Rej = {rej_target*100:.0f}%)",
              fontsize=13, fontweight='bold', color='darkgreen')
ax2.grid(alpha=0.3)
ax2.legend(loc='lower left', fontsize=11, framealpha=0.9)

ax_ins2.set_xlim(0.50, 0.62)
ax_ins2.set_ylim(0.988, 1.001)
ax_ins2.set_title("Stabilized 99% rejection line (zoom)", fontsize=9, fontweight='bold')
ax_ins2.grid(alpha=0.3)
mark_inset(ax2, ax_ins2, loc1=2, loc2=3, fc="none", ec="0.5", ls="--")

plt.tight_layout(rect=[0, 0, 1, 0.94])
# plt.savefig(f"roc_robustness_{SCENARIO}.png", dpi=200, bbox_inches='tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# CONFIG — pick the scenario to overlay

# Options:
#   "fambig"  -> overlay  _fambig_full.root
#   "momerr"  -> overlay  _momerr_x2.root
#   "both"    -> overlay  _fambigFull_momerrX2.root
SCENARIO = "momerr"

# Resolve file(s) and color/style for the scenario
# 
def perturbed_files(scenario):
    base = training_dataset_filename
    if scenario == "fambig":
        return [(base.replace('.root', '_fambig_full.root'),
                 "fambig = 1.0", "orange", "--")]
    if scenario == "momerr":
        return [(base.replace('.root', '_momerr_x2.root'),
                 "momerr × 2", "orange", "--")]
    if scenario == "both":
        return [(base.replace('.root', '_fambigFull_momerrX2.root'),
                 "Both stressed", "orange", "--")]
    raise ValueError(scenario)

perturbed = perturbed_files(SCENARIO)

# Load feature arrays for the perturbed file
x_perturbed = {}
for fname, label, color, ls in perturbed:
    x_perturbed[label] = get_features_from_file(fname)

# Figure setup
n_input_vars = len(input_var_names)
n_rows, n_cols = 3, 3
fig, axs = plt.subplots(n_rows, n_cols, figsize=(16, 9))
fig.subplots_adjust(hspace=0.5, wspace=0.3)

title_str = "Feature Distributions — Full Dataset vs " + SCENARIO.upper()
fig.suptitle(title_str, fontsize=13, fontweight='bold')

# Loop over all input variables
for i_var in range(n_input_vars):
    ax = axs.flatten()[i_var]

    # Baseline distribution
    ax.hist(x_full[:, i_var], bins=100,
            range=(x_mins[i_var], x_maxs[i_var]),
            histtype='step', color='black',
            log=log_ys[i_var],
            label=f'Full dataset (N={len(x_full):,})',
            linewidth=1.5)

    # Overlay the perturbed file
    for fname, label, color, ls in perturbed:
        x_arr = x_perturbed[label]
        # Skip if the feature is unchanged in this perturbation
        if np.array_equal(x_arr[:, i_var], x_full[:, i_var]):
            continue
        ax.hist(x_arr[:, i_var], bins=100,
                range=(x_mins[i_var], x_maxs[i_var]),
                histtype='step', color=color, ls=ls,
                log=log_ys[i_var],
                label=label, linewidth=1.4)

    ax.set_xlabel(input_var_names[i_var] + " " + units[i_var])
    ax.margins(0)
    ax.legend(fontsize=7, loc='best')

# Hide unused axes if n_input_vars < 9
for j in range(n_input_vars, n_rows * n_cols):
    axs.flatten()[j].axis('off')

# plt.savefig(f"feature_distributions_{SCENARIO}.png", dpi=150, bbox_inches='tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# CONFIG
SCENARIO = "momerr"          # "momerr" | "fambig" | "both"
FEATURE  = "momerr"          # feature name in input_var_names

# Map scenario -> file
def stressed_file(scenario):
    base = training_dataset_filename
    return {
        "momerr": (base.replace('.root', '_momerr_x2.root'),
                   "momerr × 2", "orange"),
        "fambig": (base.replace('.root', '_fambig_full.root'),
                   "fambig = 1.0", "orange"),
        "both"  : (base.replace('.root', '_fambigFull_momerrX2.root'),
                   "Both stressed", "orange"),
    }[scenario]

f_stressed, label_stressed, color_stressed = stressed_file(SCENARIO)

# Resolve feature index + load arrays
# 
i = input_var_names.index(FEATURE)

x_full    = get_features_from_file(training_dataset_filename)
x_stressed = get_features_from_file(f_stressed)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

plot_params = [
    (x_full[:, i],    'black',       f'Full dataset (N={len(x_full):,})', '-',  2.0),
    (x_stressed[:, i], color_stressed, label_stressed,                   '--', 1.5),
]

for ax in [ax1, ax2]:
    for data, color, label, style, lw in plot_params:
        ax.hist(data, bins=100,
                range=(x_mins[i], x_maxs[i]) if ax == ax1 else (0, 1.0),
                histtype='step', color=color, log=True,
                label=label, linestyle=style, linewidth=lw)
    ax.set_ylabel("Tracks")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

ax1.set_title("Full Range Distribution", fontweight='bold')
ax1.set_xlabel(f"{input_var_names[i]} {units[i]}")
ax1.margins(0)

ax2.set_title("Zoomed Distribution (0 – 1 MeV/c)", fontweight='bold')
ax2.set_xlabel(f"{FEATURE} [MeV/c]")

fig.suptitle(f"{FEATURE} Distribution — {label_stressed} vs Original",
             fontsize=14, fontweight='bold')

plt.tight_layout()
# plt.savefig(f"{FEATURE}_{SCENARIO}.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import uproot
import numpy as np
import matplotlib.pyplot as plt


SCENARIO    = "momerr"        # "fambig" | "momerr" | "both"
static_cut  = 0.92            # Original's correct cut (used in BOTH panels)
correct_cut = 0.5885          # Stressed file's correct cut (right panel only)

# ---------------------------------------------------------------------
# Resolve perturbed file
def stressed_file(scenario):
    base = training_dataset_filename
    return {
        "fambig": (base.replace('.root', '_fambig_full.root'),
                   "fambig = 1.0", "orange", "--"),
        "momerr": (base.replace('.root', '_momerr_x2.root'),
                   "momerr × 2",   "orange", "--"),
        "both"  : (base.replace('.root', '_fambigFull_momerrX2.root'),
                   "Both stressed", "crimson", "-."),
    }[scenario]

f_stressed, label_stressed, color_stressed, ls_stressed = stressed_file(SCENARIO)

# 1. Load BASELINE arrays
arrays = uproot.open(training_dataset_filename)[training_dataset_treename].arrays(library="np")

nactive     = arrays['trk.nactive']
nhits       = arrays['trk.nhits']
factive     = nactive / nhits
fambig      = arrays['trk.nnullambig'] / nactive
fstraws     = arrays['trk.nmatactive'] / nactive
t0err       = arrays['trk_ent_pars.t0err']
fitcon      = arrays['trk.fitcon']
momerr_raw  = arrays['trk_ent.momerr']

p_reco = (arrays['trk_ent.mom.fCoordinates.fX']**2 +
          arrays['trk_ent.mom.fCoordinates.fY']**2 +
          arrays['trk_ent.mom.fCoordinates.fZ']**2)**0.5
p_true = (arrays['trk_ent_mc.mom.fCoordinates.fX']**2 +
          arrays['trk_ent_mc.mom.fCoordinates.fY']**2 +
          arrays['trk_ent_mc.mom.fCoordinates.fZ']**2)**0.5
mom_res = p_reco - p_true

x_matrix = np.vstack((nactive, factive, t0err, fambig, fitcon,
                      momerr_raw, fstraws)).T
scores_orig = model.predict_proba(x_matrix)[:, 1]

# 2. Load STRESSED arrays
arrays_s = uproot.open(f_stressed)[training_dataset_treename].arrays(library="np")

nactive_s     = arrays_s['trk.nactive']
factive_s     = nactive_s / arrays_s['trk.nhits']
fambig_s      = arrays_s['trk.nnullambig'] / nactive_s
fstraws_s     = arrays_s['trk.nmatactive'] / nactive_s
t0err_s       = arrays_s['trk_ent_pars.t0err']
fitcon_s      = arrays_s['trk.fitcon']
momerr_s      = arrays_s['trk_ent.momerr']

x_stressed = np.vstack((nactive_s, factive_s, t0err_s, fambig_s,
                        fitcon_s, momerr_s, fstraws_s)).T
scores_stressed = model.predict_proba(x_stressed)[:, 1]

# 3. Plot
n_bins = 100
mom_min, mom_max = -10.0, 10.0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle(f"Momentum Resolution — Original @ 0.92 (both panels)  vs  "
             f"{label_stressed} @ 0.92 (left) and @{correct_cut} (right)",
             fontsize=18, fontweight='bold')

# Background (all tracks) on both panels
for ax in [ax1, ax2]:
    ax.hist(mom_res, bins=n_bins, range=(mom_min, mom_max),
            log=True, histtype='step', color='black',
            label='All tracks (unfiltered)', linewidth=2.0, alpha=0.7)

# Original @ static_cut = 0.92 — IDENTICAL in both panels
for ax in [ax1, ax2]:
    ax.hist(mom_res[scores_orig >= static_cut],
            bins=n_bins, range=(mom_min, mom_max),
            log=True, histtype='step', color='steelblue',
            linewidth=1.8,
            label=f'Original (Cut = {static_cut})')

# Left panel — Stressed @ SAME cut as Original (= 0.92)
ax1.hist(mom_res[scores_stressed >= static_cut],
         bins=n_bins, range=(mom_min, mom_max),
         log=True, histtype='step', color=color_stressed, linestyle=ls_stressed,
         linewidth=1.8,
         label=f'{label_stressed} (Cut = {static_cut})')

# Right panel — Stressed @ its OWN correct cut
ax2.hist(mom_res[scores_stressed >= correct_cut],
         bins=n_bins, range=(mom_min, mom_max),
         log=True, histtype='step', color=color_stressed, linestyle=ls_stressed,
         linewidth=1.8,
         label=f'{label_stressed} (Cut = {correct_cut})')

# Format
ax1.set_xlim(mom_min, mom_max)
ax1.set_xlabel("Momentum Resolution [MeV/c]", fontweight='bold', fontsize=11)
ax1.set_ylabel("Number of Tracks (Log Scale)", fontweight='bold', fontsize=11)
ax1.set_title(f"A. Both files at Static Cut = {static_cut}",
              fontsize=13, fontweight='bold', color='crimson')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='upper right', fontsize=11, framealpha=0.9)

ax2.set_xlim(mom_min, mom_max)
ax2.set_xlabel("Momentum Resolution [MeV/c]", fontweight='bold', fontsize=11)
ax2.set_ylabel("Number of Tracks (Log Scale)", fontweight='bold', fontsize=11)
ax2.set_title(f"B. Original @ {static_cut} vs {label_stressed} @ {correct_cut}",
              fontsize=13, fontweight='bold', color='darkgreen')
ax2.grid(True, alpha=0.3)
ax2.legend(loc='upper right', fontsize=11, framealpha=0.9)

plt.tight_layout()
# plt.savefig(f"mom_resolution_{SCENARIO}.png", dpi=200, bbox_inches='tight')
plt.show()


## Both Fambig and Momerr Combined 

In [ ]:
# CONFIG — change this one variable per run

# Options:
#   "fambig"        ->  tests  _fambig_full.root
#   "momerr"        ->  tests  _momerr_x2.root
#   "both"          ->  tests  _fambigFull_momerrX2.root
#   "momerr_pm50"   ->  tests both  _momerr_plus50pct.root  AND  _momerr_minus50pct.root

SCENARIO = "both" 

import numpy as np
from sklearn.metrics import roc_curve, auc

# Map scenario -> file(s)

def files_for(scenario):
    base = training_dataset_filename
    if scenario == "fambig":
        return [("fambig = 1.0",  base.replace('.root', '_fambig_full.root'))]
    if scenario == "momerr":
        return [("momerr × 2",    base.replace('.root', '_momerr_x2.root'))]
    if scenario == "both":
        return [("Both stressed", base.replace('.root', '_fambigFull_momerrX2.root'))]
    if scenario == "momerr_pm50":
        return [("momerr +50%", base.replace('.root', '_momerr_plus50pct.root')),
                ("momerr -50%", base.replace('.root', '_momerr_minus50pct.root'))]
    raise ValueError(f"Unknown scenario: {scenario}")

# ---------------------------------------------------------------------
# Header
# ---------------------------------------------------------------------
print(f"{'Dataset':<22} {'AUC':>8}  {'Sig eff':>10}  {'BDT cut':>9}  "
      f"{'Mean shift':>12}  {'% sig flipped':>14}  {'Δsig':>7}  {'Δbkg':>7}")
print("-" * 110)

# ---------------------------------------------------------------------
# Baseline numbers (computed once)
# ---------------------------------------------------------------------
fpr0, tpr0, thr0 = roc_curve(y_full, y_pred_original_full, pos_label=1)
idx0    = np.where(1 - fpr0 >= 0.99)[0]
cut_idx = idx0[-1] if len(idx0) > 0 else 0
bdt_cut = thr0[cut_idx]
print(f"{'Baseline (orig)':<22} {auc(fpr0,tpr0):>8.4f}  "
      f"{tpr0[cut_idx]*100:>9.2f}%  {bdt_cut:>9.4f}  "
      f"{0.0:>12.6f}  {0.0:>13.2f}%  {0:>7}  {0:>7}")

# ---------------------------------------------------------------------
# Loop over the file(s) for this scenario
# ---------------------------------------------------------------------
for label, fname in files_for(SCENARIO):
    # 1) Get features for this stressed file
    x_new = get_features_from_file(fname)

    # 2) Run original model — no retraining
    y_new = model.predict_proba(x_new)[:, 1]

    # 3) Per-track prediction shift
    delta = y_new - y_pred_original_full

    # 4) ROC + AUC
    fpr, tpr, thr = roc_curve(y_full, y_new, pos_label=1)
    auc_score     = auc(fpr, tpr)

    # 5) Optimal cut at 99% BG rejection
    idx        = np.where(1 - fpr >= 0.99)[0]
    cut_i      = idx[-1] if len(idx) > 0 else 0
    sig_eff    = tpr[cut_i]
    opt_cut    = thr[cut_i]

    # 6) Track flipping at the ORIGINAL cut (bdt_cut)
    pass_orig  = y_pred_original_full >= bdt_cut
    pass_new   = y_new                >= bdt_cut
    flipped    = (pass_orig != pass_new)
    sig_flipped = (flipped & (y_full == 1)).sum()
    bkg_flipped = (flipped & (y_full == 0)).sum()
    pct_sig     = sig_flipped / N_signal * 100.0

    print(f"{label:<22} {auc_score:>8.4f}  {sig_eff*100:>9.2f}%  "
          f"{opt_cut:>9.4f}  {delta.mean():>+12.6f}  "
          f"{pct_sig:>13.2f}%  {sig_flipped:>7}  {bkg_flipped:>7}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, auc

# CONFIG — change to "fambig", "momerr", "both", or "momerr_pm50"

SCENARIO = "both"

# Map scenario -> list of (label, color, scores_array)
# (the scores arrays are produced further down — this is just metadata)

def scenario_meta(s):
    if s == "fambig":
        return {
            "title":    "BDT Score: Original vs fambig = 1.0",
            "rows":     [("Original",      "steelblue", 2.0),
                         ("fambig = 1.0",  "orange",    1.5)],
            "ylim":     (0.85, 1.0),
        }
    if s == "momerr":
        return {
            "title":    "BDT Score: Original vs momerr × 2",
            "rows":     [("Original",      "steelblue", 2.0),
                         ("momerr × 2",    "orange",    1.5)],
            "ylim":     (0.85, 1.0),
        }
    if s == "both":
        return {
            "title":    "BDT Score: Original vs Both stressed (fambig=1 & momerr×2)",
            "rows":     [("Original",       "steelblue", 2.0),
                         ("Both stressed",  "orange",    1.5)],
            "ylim":     (0.85, 1.0),
        }
    if s == "momerr_pm50":
        return {
            "title":    "BDT Score: Original vs momerr Heavy Perturbations (±50%)",
            "rows":     [("Original",      "steelblue", 2.0),
                         ("momerr +50%",   "orange",    1.5),
                         ("momerr -50%",   "darkblue",  1.5)],
            "ylim":     (0.85, 1.0),
        }
    raise ValueError(s)

meta = scenario_meta(SCENARIO)

# ---------------------------------------------------------------------
# Resolve filenames for this scenario
# ---------------------------------------------------------------------
def stress_filename(scenario, variant):
    base = training_dataset_filename
    return {
        "fambig_full"      : base.replace('.root', '_fambig_full.root'),
        "momerr_x2"        : base.replace('.root', '_momerr_x2.root'),
        "both"             : base.replace('.root', '_fambigFull_momerrX2.root'),
        "momerr_plus50"    : base.replace('.root', '_momerr_plus50pct.root'),
        "momerr_minus50"   : base.replace('.root', '_momerr_minus50pct.root'),
    }[variant]

# ---------------------------------------------------------------------
# Build the perturbed score arrays for this scenario
# ---------------------------------------------------------------------
def scores_for(scenario):
    if scenario == "fambig":
        x = get_features_from_file(stress_filename(scenario, "fambig_full"))
        return model.predict_proba(x)[:, 1]
    if scenario == "momerr":
        x = get_features_from_file(stress_filename(scenario, "momerr_x2"))
        return model.predict_proba(x)[:, 1]
    if scenario == "both":
        x = get_features_from_file(stress_filename(scenario, "both"))
        return model.predict_proba(x)[:, 1]
    if scenario == "momerr_pm50":
        x_up   = get_features_from_file(stress_filename(scenario, "momerr_plus50"))
        x_down = get_features_from_file(stress_filename(scenario, "momerr_minus50"))
        return model.predict_proba(x_up)[:, 1], model.predict_proba(x_down)[:, 1]

if SCENARIO == "momerr_pm50":
    y_new_up, y_new_down = scores_for(SCENARIO)
    perturbed = {
        "momerr +50%" : y_new_up,
        "momerr -50%" : y_new_down,
    }
else:
    y_new = scores_for(SCENARIO)
    perturbed = {meta["rows"][1][0]: y_new}

# ---------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
fig.suptitle(meta["title"], fontsize=16, fontweight='bold')

# Build the dataset list (always include Original at front)
datasets = [(y_pred_original_full, "Original", "steelblue", 2.0)]
for label, color, lw in meta["rows"][1:]:
    datasets.append((perturbed[label], label, color, lw))

for ax, panel_title, xlim in zip(
        axes,
        ["Full range (log scale)", "Zoomed around cut"],
        [None, meta["ylim"]]):

    for scores, label, color, lw in datasets:
        ax.hist(scores, bins=100, histtype="step",
                linewidth=lw, color=color, label=label)

    # BASELINE cut (black dashed)
    ax.axvline(trkqual_cut, color='black', linestyle='--', linewidth=1.5,
               label=f'Baseline cut = {trkqual_cut:.4f}')

    #  CORRECTED cut per dataset at 99% BG rejection 
    # solid thin vertical line for each perturbed file
    palette = {"Original": "steelblue"}
    for i, (label, color, _) in enumerate(meta["rows"][1:]):
        scores = perturbed[label]
        fpr, tpr, thr = roc_curve(y_full, scores, pos_label=1)
        idx = np.where(1 - fpr >= 0.99)[0]
        if len(idx) > 0:
            ci   = idx[-1]
            cut  = thr[ci]
            ax.axvline(cut, color=color, linestyle=':', linewidth=1.5,
                       label=f'{label} cut (99% BG rej) = {cut:.4f}')

    ax.set_xlabel("BDT Score")
    ax.set_ylabel("Tracks")
    ax.set_title(panel_title)
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(alpha=0.3)
    if xlim:
        ax.set_xlim(xlim)
    else:
        ax.set_yscale('log')

# Build summary table

N_signal = (y_full == 1).sum()
bdt_pass_orig = y_pred_original_full >= trkqual_cut

table_data  = [["Original Baseline", "—", "—", "—", "—", "—"]]
columns     = ["Configuration", "Mean Shift", "Changed", "% of Signal", "Signal", "Bkg"]

for label, color, _ in meta["rows"][1:]:
    y_new       = perturbed[label]
    delta       = y_new - y_pred_original_full
    bdt_pass    = y_new >= trkqual_cut
    changed     = (bdt_pass_orig != bdt_pass).sum()
    sig_changed = ((bdt_pass_orig != bdt_pass) & (y_full == 1)).sum()
    bkg_changed = ((bdt_pass_orig != bdt_pass) & (y_full == 0)).sum()
    table_data.append([
        label,
        f"{delta.mean():+.6f}",
        f"{changed:,}",
        f"{sig_changed/N_signal*100:.2f}%",
        f"{sig_changed:,}",
        f"{bkg_changed:,}",
    ])

# Render the table below the plots

n_rows = len(table_data) + 1
table_ax = fig.add_axes([0.05, -0.04 - 0.05*n_rows, 0.9, 0.05*n_rows])
table_ax.axis('off')

summary_table = table_ax.table(cellText=table_data, colLabels=columns,
                               loc='center', cellLoc='center')
summary_table.auto_set_font_size(False)
summary_table.set_fontsize(12)
summary_table.scale(1.0, 1.6)

for (row, col), cell in summary_table.get_celld().items():
    if col == 0 and row > 0:
        cell.set_text_props(ha='left')
    if row == 0:
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#2f4f4f')

plt.subplots_adjust(bottom=0.18 + 0.05*n_rows)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, auc
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

# CONFIG — change this to switch scenarios

# Options:
#   "fambig"   ->  just fambig = 1.0
#   "momerr"   ->  just momerr × 2
#   "both"     ->  combined fambig=1 + momerr×2
#   "all"      ->  momerr ±10%, ±50%, fambig=1, combined, original

SCENARIO = "both"


# Map scenario -> list of (filename, label, color)

def datasets_for(scenario):
    base = training_dataset_filename

    common = [
        (base, "Original", "steelblue"),
    ]
    perturbed = {
        "fambig" : [(base.replace('.root', '_fambig_full.root'),
                     "fambig = 1.0", "orange")],
        "momerr" : [(base.replace('.root', '_momerr_x2.root'),
                     "momerr × 2", "orange")],
        "both"   : [(base.replace('.root', '_fambigFull_momerrX2.root'),
                     "Both stressed", "orange")],
        "all"    : [
            (base.replace('.root', '_momerr_plus10pct.root'),
             "momerr +10%", "tomato"),
            (base.replace('.root', '_momerr_minus10pct.root'),
             "momerr -10%", "limegreen"),
            (base.replace('.root', '_momerr_plus50pct.root'),
             "momerr +50%", "orange"),
            (base.replace('.root', '_momerr_minus50pct.root'),
             "momerr -50%", "darkblue"),
            (base.replace('.root', '_fambig_full.root'),
             "fambig = 1.0", "purple"),
            (base.replace('.root', '_fambigFull_momerrX2.root'),
             "Both stressed", "crimson"),
        ],
    }
    return common + perturbed[scenario]

dataset_meta = datasets_for(SCENARIO)


# Load / compute (score, label) for each entry

def load_scores(fname):
    x = get_features_from_file(fname)
    return model.predict_proba(x)[:, 1]

datasets_roc = []
for fname, label, color in dataset_meta:
    scores = (y_pred_original_full if fname == training_dataset_filename
              else load_scores(fname))
    datasets_roc.append((scores, y_full, label, color))


# Plot parameters

static_cut_target = 0.92
rej_target        = 0.99

# Initialize side-by-side subplots

n_panels = len(datasets_roc)
fig_w = max(22, 14 + 1.5 * (n_panels - 4))   # wider if many panels
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(fig_w, 10))
fig.suptitle(f"ROC Curve Analysis — Static vs Adaptive Cut  "
             f"(scenario: {SCENARIO})",
             fontsize=22, fontweight='bold', y=0.96)

ax_ins1 = inset_axes(ax1, width="40%", height="40%", loc='upper right',
                     bbox_to_anchor=(-0.02, -0.02, 1, 1),
                     bbox_transform=ax1.transAxes)
ax_ins2 = inset_axes(ax2, width="40%", height="40%", loc='upper right',
                     bbox_to_anchor=(-0.02, -0.02, 1, 1),
                     bbox_transform=ax2.transAxes)

# Loop over datasets

for scores, labels, name, color in datasets_roc:
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    auc_score = auc(fpr, tpr)

    #  LEFT PANEL: STATIC CUT 
    idx_static = np.where(thresholds >= static_cut_target)[0]
    idx_s = idx_static[-1] if len(idx_static) > 0 else 0
    sig_eff_s = float(tpr[idx_s])
    bkg_rej_s = float(1 - fpr[idx_s])

    ax1.plot(tpr, 1 - fpr, color=color, linewidth=2,
             label=f"{name:<14} | AUC={auc_score:.4f} | "
                   f"Eff={sig_eff_s*100:5.1f}% | "
                   f"Rej={bkg_rej_s*100:5.2f}%")
    ax1.plot(tpr[idx_s], 1 - fpr[idx_s], 'o', color=color, markersize=8)
    ax_ins1.plot(tpr, 1 - fpr, color=color, linewidth=1.5)
    ax_ins1.plot(tpr[idx_s], 1 - fpr[idx_s], 'o', color=color, markersize=6)

    #  RIGHT PANEL: ADAPTIVE CUT 
    idx_dyn = np.where(1 - fpr >= rej_target)[0]
    idx_d = idx_dyn[-1] if len(idx_dyn) > 0 else 0
    sig_eff_d = float(tpr[idx_d])
    opt_cut   = float(thresholds[idx_d])

    ax2.plot(tpr, 1 - fpr, color=color, linewidth=2,
             label=f"{name:<14} | AUC={auc_score:.4f} | "
                   f"Cut={opt_cut:.3f} | "
                   f"Eff={sig_eff_d*100:5.1f}%")
    ax2.plot(tpr[idx_d], 1 - fpr[idx_d], 'o', color=color, markersize=8)
    ax_ins2.plot(tpr, 1 - fpr, color=color, linewidth=1.5)
    ax_ins2.plot(tpr[idx_d], 1 - fpr[idx_d], 'o', color=color, markersize=6)


# Format LEFT subplot

ax1.plot([0, 1], [1, 0], 'k--', alpha=0.3)
ax1.set_xlabel("Signal Efficiency (TPR)", fontsize=12, fontweight='bold')
ax1.set_ylabel("Background Rejection (1 − FPR)", fontsize=12, fontweight='bold')
ax1.set_title(f"A. Operating Points at Static BDT Cut = {static_cut_target:.2f}",
              fontsize=13, fontweight='bold', color='crimson')
ax1.grid(alpha=0.3)
ax1.legend(loc='lower left', fontsize=11, framealpha=0.9)

ax_ins1.set_xlim(0.35, 0.82)
ax_ins1.set_ylim(0.915, 1.002)
ax_ins1.set_title("Operating-point scatter (zoom)", fontsize=9, fontweight='bold')
ax_ins1.grid(alpha=0.3)
mark_inset(ax1, ax_ins1, loc1=2, loc2=3, fc="none", ec="0.5", ls="--")

# ---------------------------------------------------------------------
# Format RIGHT subplot
# ---------------------------------------------------------------------
ax2.plot([0, 1], [1, 0], 'k--', alpha=0.3)
ax2.set_xlabel("Signal Efficiency (TPR)", fontsize=12, fontweight='bold')
ax2.set_ylabel("Background Rejection (1 − FPR)", fontsize=12, fontweight='bold')
ax2.set_title(f"B. Operating Points at Adaptive Cut (Rej = {rej_target*100:.0f}%)",
              fontsize=13, fontweight='bold', color='darkgreen')
ax2.grid(alpha=0.3)
ax2.legend(loc='lower left', fontsize=11, framealpha=0.9)

ax_ins2.set_xlim(0.50, 0.62)
ax_ins2.set_ylim(0.988, 1.001)
ax_ins2.set_title("Stabilized 99% rejection line (zoom)", fontsize=9, fontweight='bold')
ax_ins2.grid(alpha=0.3)
mark_inset(ax2, ax_ins2, loc1=2, loc2=3, fc="none", ec="0.5", ls="--")

plt.tight_layout(rect=[0, 0, 1, 0.94])
# plt.savefig(f"roc_robustness_{SCENARIO}.png", dpi=200, bbox_inches='tight")
plt.show()


In [ ]:
import uproot
import numpy as np
import matplotlib.pyplot as plt

# CONFIG
SCENARIO    = "both"        # "fambig" | "momerr" | "both"
static_cut  = 0.92            # Original's correct cut (used in BOTH panels)
correct_cut = 0.3311          # Stressed file's correct cut (right panel only)

# Resolve perturbed file

def stressed_file(scenario):
    base = training_dataset_filename
    return {
        "fambig": (base.replace('.root', '_fambig_full.root'),
                   "fambig = 1.0", "orange", "--"),
        "momerr": (base.replace('.root', '_momerr_x2.root'),
                   "momerr × 2",   "orange", "--"),
        "both"  : (base.replace('.root', '_fambigFull_momerrX2.root'),
                   "Both stressed", "crimson", "-."),
    }[scenario]

f_stressed, label_stressed, color_stressed, ls_stressed = stressed_file(SCENARIO)


# 1. Load BASELINE arrays
arrays = uproot.open(training_dataset_filename)[training_dataset_treename].arrays(library="np")

nactive     = arrays['trk.nactive']
nhits       = arrays['trk.nhits']
factive     = nactive / nhits
fambig      = arrays['trk.nnullambig'] / nactive
fstraws     = arrays['trk.nmatactive'] / nactive
t0err       = arrays['trk_ent_pars.t0err']
fitcon      = arrays['trk.fitcon']
momerr_raw  = arrays['trk_ent.momerr']

p_reco = (arrays['trk_ent.mom.fCoordinates.fX']**2 +
          arrays['trk_ent.mom.fCoordinates.fY']**2 +
          arrays['trk_ent.mom.fCoordinates.fZ']**2)**0.5
p_true = (arrays['trk_ent_mc.mom.fCoordinates.fX']**2 +
          arrays['trk_ent_mc.mom.fCoordinates.fY']**2 +
          arrays['trk_ent_mc.mom.fCoordinates.fZ']**2)**0.5
mom_res = p_reco - p_true

x_matrix = np.vstack((nactive, factive, t0err, fambig, fitcon,
                      momerr_raw, fstraws)).T
scores_orig = model.predict_proba(x_matrix)[:, 1]

# 2. Load STRESSED arrays

arrays_s = uproot.open(f_stressed)[training_dataset_treename].arrays(library="np")

nactive_s     = arrays_s['trk.nactive']
factive_s     = nactive_s / arrays_s['trk.nhits']
fambig_s      = arrays_s['trk.nnullambig'] / nactive_s
fstraws_s     = arrays_s['trk.nmatactive'] / nactive_s
t0err_s       = arrays_s['trk_ent_pars.t0err']
fitcon_s      = arrays_s['trk.fitcon']
momerr_s      = arrays_s['trk_ent.momerr']

x_stressed = np.vstack((nactive_s, factive_s, t0err_s, fambig_s,
                        fitcon_s, momerr_s, fstraws_s)).T
scores_stressed = model.predict_proba(x_stressed)[:, 1]

# ---------------------------------------------------------------------
# 3. Plot
# ---------------------------------------------------------------------
n_bins = 100
mom_min, mom_max = -10.0, 10.0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle(f"Momentum Resolution — Original @ 0.92 (both panels)  vs  "
             f"{label_stressed} @ 0.92 (left) and @{correct_cut} (right)",
             fontsize=18, fontweight='bold')

# Background (all tracks) on both panels
for ax in [ax1, ax2]:
    ax.hist(mom_res, bins=n_bins, range=(mom_min, mom_max),
            log=True, histtype='step', color='black',
            label='All tracks (unfiltered)', linewidth=2.0, alpha=0.7)

# Original @ static_cut = 0.92 — IDENTICAL in both panels
for ax in [ax1, ax2]:
    ax.hist(mom_res[scores_orig >= static_cut],
            bins=n_bins, range=(mom_min, mom_max),
            log=True, histtype='step', color='steelblue',
            linewidth=1.8,
            label=f'Original (Cut = {static_cut})')

# Left panel — Stressed @ SAME cut as Original (= 0.92)
ax1.hist(mom_res[scores_stressed >= static_cut],
         bins=n_bins, range=(mom_min, mom_max),
         log=True, histtype='step', color=color_stressed, linestyle=ls_stressed,
         linewidth=1.8,
         label=f'{label_stressed} (Cut = {static_cut})')

# Right panel — Stressed @ its OWN correct cut
ax2.hist(mom_res[scores_stressed >= correct_cut],
         bins=n_bins, range=(mom_min, mom_max),
         log=True, histtype='step', color=color_stressed, linestyle=ls_stressed,
         linewidth=1.8,
         label=f'{label_stressed} (Cut = {correct_cut})')

# Format
ax1.set_xlim(mom_min, mom_max)
ax1.set_xlabel("Momentum Resolution [MeV/c]", fontweight='bold', fontsize=11)
ax1.set_ylabel("Number of Tracks (Log Scale)", fontweight='bold', fontsize=11)
ax1.set_title(f"A. Both files at Static Cut = {static_cut}",
              fontsize=13, fontweight='bold', color='crimson')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='upper right', fontsize=11, framealpha=0.9)

ax2.set_xlim(mom_min, mom_max)
ax2.set_xlabel("Momentum Resolution [MeV/c]", fontweight='bold', fontsize=11)
ax2.set_ylabel("Number of Tracks (Log Scale)", fontweight='bold', fontsize=11)
ax2.set_title(f"B. Original @ {static_cut} vs {label_stressed} @ {correct_cut}",
              fontsize=13, fontweight='bold', color='darkgreen')
ax2.grid(True, alpha=0.3)
ax2.legend(loc='upper right', fontsize=11, framealpha=0.9)

plt.tight_layout()
# plt.savefig(f"mom_resolution_{SCENARIO}.png", dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from sklearn.metrics import roc_curve, roc_auc_score


# CONFIG
SCENARIO    = "both"          # "fambig" | "momerr" | "both"
static_cut  = 0.92
correct_cut = 0.3311          # Stressed file's correct cut


# Resolve perturbed file

def stressed_file(scenario):
    base = training_dataset_filename
    return {
        "fambig": (base.replace('.root', '_fambig_full.root'),
                   "fambig = 1.0", "orange", "--"),
        "momerr": (base.replace('.root', '_momerr_x2.root'),
                   "momerr × 2",   "orange", "--"),
        "both"  : (base.replace('.root', '_fambigFull_momerrX2.root'),
                   "Both stressed", "crimson", "-."),
    }[scenario]

f_stressed, label_stressed, color_stressed, ls_stressed = stressed_file(SCENARIO)

# 1. Load BASELINE arrays
arrays = uproot.open(training_dataset_filename)[training_dataset_treename].arrays(library="np")

nactive     = arrays['trk.nactive']
nhits       = arrays['trk.nhits']
factive     = nactive / nhits
fambig      = arrays['trk.nnullambig'] / nactive
fstraws     = arrays['trk.nmatactive'] / nactive
t0err       = arrays['trk_ent_pars.t0err']
fitcon      = arrays['trk.fitcon']
momerr_raw  = arrays['trk_ent.momerr']

p_reco = (arrays['trk_ent.mom.fCoordinates.fX']**2 +
          arrays['trk_ent.mom.fCoordinates.fY']**2 +
          arrays['trk_ent.mom.fCoordinates.fZ']**2)**0.5
p_true = (arrays['trk_ent_mc.mom.fCoordinates.fX']**2 +
          arrays['trk_ent_mc.mom.fCoordinates.fY']**2 +
          arrays['trk_ent_mc.mom.fCoordinates.fZ']**2)**0.5
mom_res = p_reco - p_true

x_matrix = np.vstack((nactive, factive, t0err, fambig, fitcon,
                      momerr_raw, fstraws)).T
scores_orig = model.predict_proba(x_matrix)[:, 1]


# 2. Load STRESSED arrays
arrays_s = uproot.open(f_stressed)[training_dataset_treename].arrays(library="np")

nactive_s     = arrays_s['trk.nactive']
factive_s     = nactive_s / arrays_s['trk.nhits']
fambig_s      = arrays_s['trk.nnullambig'] / nactive_s
fstraws_s     = arrays_s['trk.nmatactive'] / nactive_s
t0err_s       = arrays_s['trk_ent_pars.t0err']
fitcon_s      = arrays_s['trk.fitcon']
momerr_s      = arrays_s['trk_ent.momerr']

x_stressed = np.vstack((nactive_s, factive_s, t0err_s, fambig_s,
                        fitcon_s, momerr_s, fstraws_s)).T
scores_stressed = model.predict_proba(x_stressed)[:, 1]

# 3. Figure layout: 2 rows × 3 columns
fig = plt.figure(figsize=(22, 12))
gs  = GridSpec(2, 3, height_ratios=[1, 1], hspace=0.30, wspace=0.28)

ax_score   = fig.add_subplot(gs[0, 0])   # BDT score distribution
ax_mom     = fig.add_subplot(gs[0, 1])   # momentum resolution (static)
ax_momdyn  = fig.add_subplot(gs[0, 2])   # momentum resolution (dynamic)
ax_resid   = fig.add_subplot(gs[1, 0])   # momentum residual
ax_flip    = fig.add_subplot(gs[1, 1])   # flip analysis
ax_eff     = fig.add_subplot(gs[1, 2])   # signal efficiency vs cut

fig.suptitle(
    f"BDT Breakage Catalog  —  {label_stressed}  (scenario: {SCENARIO})",
    fontsize=20, fontweight='bold')

# PANEL 1: BDT score distribution
bins_score = np.linspace(0, 1, 80)
ax_score.hist(scores_orig,    bins=bins_score, histtype='stepfilled',
              alpha=0.35, color='steelblue', label=f'Original  (mean={scores_orig.mean():.3f})')
ax_score.hist(scores_stressed, bins=bins_score, histtype='step',
              color=color_stressed, lw=1.8, linestyle=ls_stressed,
              label=f'{label_stressed}  (mean={scores_stressed.mean():.3f})')
ax_score.axvline(static_cut,  color='black',    ls='--', lw=1.5, label=f'Static cut = {static_cut}')
ax_score.axvline(correct_cut, color=color_stressed, ls=':',  lw=1.5, label=f'Dynamic cut = {correct_cut}')
ax_score.set_xlabel("BDT score"); ax_score.set_ylabel("Tracks")
ax_score.set_title("1. Score distribution (score collapse?)", fontweight='bold')
ax_score.legend(fontsize=9, loc='upper center'); ax_score.grid(alpha=0.3)

# PANELS 2 & 3: Momentum resolution (static & dynamic)

n_bins   = 100
mom_min, mom_max = -10.0, 10.0
bin_edges   = np.linspace(mom_min, mom_max, n_bins + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

for ax, mask_orig, mask_stress, cut_orig, cut_stress, title in [
    (ax_mom,    scores_orig    >= static_cut,  scores_stressed >= static_cut,
     static_cut, static_cut,  f"2. Mom res @ static cut = {static_cut}"),
    (ax_momdyn, scores_orig    >= static_cut,  scores_stressed >= correct_cut,
     static_cut, correct_cut, f"3. Mom res @ dynamic cut = {correct_cut}"),
]:
    ax.hist(mom_res, bins=bin_edges, log=True,
            histtype='step', color='black', alpha=0.6,
            label='All tracks', linewidth=1.5)
    ax.hist(mom_res[mask_orig], bins=bin_edges, log=True,
            histtype='step', color='steelblue', linewidth=1.8,
            label=f'Original (cut {cut_orig})')
    ax.hist(mom_res[mask_stress], bins=bin_edges, log=True,
            histtype='step', color=color_stressed, linestyle=ls_stressed,
            linewidth=1.8, label=f'{label_stressed} (cut {cut_stress})')
    ax.set_xlabel("Momentum Resolution [MeV/c]"); ax.set_ylabel("Tracks (log)")
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=9, loc='upper right'); ax.grid(alpha=0.3)

# PANEL 4: Per-track momentum residual
mask_stress_static   = scores_stressed >= static_cut
mask_orig_static     = scores_orig    >= static_cut
ax_resid.scatter(mom_res[mask_orig_static & ~mask_stress_static],
                 scores_stressed[mask_orig_static & ~mask_stress_static],
                 s=0.5, alpha=0.3, color='orange', label='Lost (orig passed, stress failed)')
ax_resid.scatter(mom_res[mask_stress_static & ~mask_orig_static],
                 scores_stressed[mask_stress_static & ~mask_orig_static],
                 s=0.5, alpha=0.3, color='green', label='Gained')
ax_resid.axhline(static_cut, color='black', ls='--', lw=1.2)
ax_resid.set_xlabel("Momentum Resolution [MeV/c]")
ax_resid.set_ylabel("BDT score (stressed)")
ax_resid.set_xlim(-10, 10); ax_resid.set_ylim(0, 1)
ax_resid.set_title("4. Where are the lost tracks? (per-track)", fontweight='bold')
ax_resid.legend(fontsize=9, loc='lower right'); ax_resid.grid(alpha=0.3)

# PANEL 5: Flip analysis
n_total = len(scores_orig)
n_lost_signal = (mask_orig_static & ~mask_stress_static).sum()
n_gained     = (mask_stress_static & ~mask_orig_static).sum()
n_pass_orig  = mask_orig_static.sum()
n_pass_stress = mask_stress_static.sum()

categories = ['Pass @ orig cut', 'Pass @ stress cut', 'Flip LOST (orig → fail)',
              'Flip GAINED (fail → pass)']
counts = [n_pass_orig, n_pass_stress, n_lost_signal, n_gained]
colors_b = ['steelblue', color_stressed, 'crimson', 'limegreen']

bars = ax_flip.bar(categories, counts, color=colors_b, edgecolor='black')
for bar, c in zip(bars, counts):
    ax_flip.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f'{c:,}\n({c/n_total*100:.1f}%)',
                 ha='center', va='bottom', fontsize=9, fontweight='bold')
ax_flip.set_xticklabels(categories, rotation=15, ha='right', fontsize=9)
ax_flip.set_ylabel("Number of tracks")
ax_flip.set_title("5. Track pass/fail flip (at static cut)", fontweight='bold')
ax_flip.grid(alpha=0.3, axis='y')


# PANEL 6: Signal efficiency vs cut

cuts = np.linspace(0, 1, 200)
eff_orig   = np.array([(scores_orig    >= c).mean() for c in cuts])
eff_stress = np.array([(scores_stressed >= c).mean() for c in cuts])
ax_eff.plot(cuts, eff_orig,   color='steelblue',       lw=2, label='Original')
ax_eff.plot(cuts, eff_stress, color=color_stressed,    lw=2, linestyle=ls_stressed,
            label=label_stressed)
ax_eff.axvline(static_cut,  color='black', ls='--', lw=1.2, label=f'Static = {static_cut}')
ax_eff.axvline(correct_cut, color=color_stressed, ls=':', lw=1.2,
               label=f'Dynamic = {correct_cut}')
# Mark efficiency at the two cuts
ax_eff.plot([static_cut, correct_cut],
            [eff_orig[np.argmin(np.abs(cuts - static_cut))],
             eff_stress[np.argmin(np.abs(cuts - correct_cut))]],
            'o', color='black', markersize=8)
ax_eff.set_xlabel("BDT cut")
ax_eff.set_ylabel("Fraction of tracks above cut")
ax_eff.set_title("6. Cumulative score distribution", fontweight='bold')
ax_eff.legend(fontsize=9, loc='upper right'); ax_eff.grid(alpha=0.3)
ax_eff.set_xlim(0, 1); ax_eff.set_ylim(0, 1.02)


# Print breakage summary
print("\n" + "=" * 78)
print(f"  BDT BREAKAGE CATALOG  —  scenario: {SCENARIO}  ({label_stressed})")
print("=" * 78)
print(f"  Total tracks                  : {n_total:,}")
print(f"  Mean score (orig / stress)     : {scores_orig.mean():.4f}  /  {scores_stressed.mean():.4f}   (Δ = {scores_stressed.mean() - scores_orig.mean():+.4f})")
print(f"  Median score (orig / stress)  : {np.median(scores_orig):.4f}  /  {np.median(scores_stressed):.4f}")
print(f"  Std score  (orig / stress)    : {scores_orig.std():.4f}  /  {scores_stressed.std():.4f}")
print(f"  Pass rate @ static cut        : orig = {n_pass_orig/n_total*100:5.2f}%,  "
      f"stress = {n_pass_stress/n_total*100:5.2f}%   "
      f"(Δ = {(n_pass_stress - n_pass_orig)/n_total*100:+.2f} pp)")
print(f"  Tracks LOST at static cut     : {n_lost_signal:,}  ({n_lost_signal/n_total*100:.2f}%)")
print(f"  Tracks GAINED at static cut   : {n_gained:,}  ({n_gained/n_total*100:.2f}%)")
print(f"  Required cut shift (→ 99% rej) : {correct_cut - static_cut:+.4f}")

# Tail bin analysis
counts_orig_t   = np.histogram(mom_res[mask_orig_static],   bins=bin_edges)[0]
counts_stress_t = np.histogram(mom_res[mask_stress_static], bins=bin_edges)[0]
nz = counts_orig_t > 0
fd = np.zeros_like(counts_stress_t, dtype=float)
fd[nz] = (counts_stress_t[nz] - counts_orig_t[nz]) / counts_orig_t[nz] * 100
tail_mask = (bin_centers >= 2.5) & (bin_centers <= 4.0)
print(f"\n  High-resolution tail (+2.5 to +4.0 MeV/c) at STATIC cut:")
print(f"    mean Δ = {fd[tail_mask].mean():+.2f}%,   max |Δ| = {np.abs(fd[tail_mask]).max():.2f}%")

# Same at dynamic
counts_stress_t2 = np.histogram(mom_res[mask_stress_static & (scores_stressed >= correct_cut)],
                                bins=bin_edges)[0] if False else \
                   np.histogram(mom_res[scores_stressed >= correct_cut], bins=bin_edges)[0]
nz2 = counts_orig_t > 0
fd2 = np.zeros_like(counts_stress_t2, dtype=float)
fd2[nz2] = (counts_stress_t2[nz2] - counts_orig_t[nz2]) / counts_orig_t[nz2] * 100
print(f"  High-resolution tail (+2.5 to +4.0 MeV/c) at DYNAMIC cut:")
print(f"    mean Δ = {fd2[tail_mask].mean():+.2f}%,   max |Δ| = {np.abs(fd2[tail_mask]).max():.2f}%")
print("=" * 78)

plt.tight_layout(rect=[0, 0, 1, 0.96])
# plt.savefig(f"breakage_catalog_{SCENARIO}.png", dpi=180, bbox_inches='tight')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, auc
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

# CONFIG — change this to switch scenarios

# Options:
#   "fambig"   ->  just fambig = 1.0
#   "momerr"   ->  just momerr × 2
#   "both"     ->  combined fambig=1 + momerr×2
#   "all"      ->  momerr ±10%, ±50%, fambig=1, combined, original
SCENARIO = "all"


# Map scenario -> list of (filename, label, color)
def datasets_for(scenario):
    base = training_dataset_filename

    common = [
        (base, "Original", "steelblue"),
    ]
    perturbed = {
        "fambig" : [(base.replace('.root', '_fambig_full.root'),
                     "fambig = 1.0", "orange")],
        "momerr" : [(base.replace('.root', '_momerr_x2.root'),
                     "momerr × 2", "orange")],
        "both"   : [(base.replace('.root', '_fambigFull_momerrX2.root'),
                     "Both stressed", "orange")],
        "all"    : [
            (base.replace('.root', '_momerr_plus10pct.root'),
             "momerr +10%", "tomato"),
            (base.replace('.root', '_momerr_minus10pct.root'),
             "momerr -10%", "limegreen"),
            (base.replace('.root', '_momerr_plus50pct.root'),
             "momerr +50%", "orange"),
            (base.replace('.root', '_momerr_minus50pct.root'),
             "momerr -50%", "darkblue"),
            (base.replace('.root', '_fambig_full.root'),
             "fambig = 1.0", "purple"),
            (base.replace('.root', '_fambigFull_momerrX2.root'),
             "Both stressed", "crimson"),
        ],
    }
    return common + perturbed[scenario]

dataset_meta = datasets_for(SCENARIO)

# Load / compute (score, label) for each entry

def load_scores(fname):
    x = get_features_from_file(fname)
    return model.predict_proba(x)[:, 1]

datasets_roc = []
for fname, label, color in dataset_meta:
    scores = (y_pred_original_full if fname == training_dataset_filename
              else load_scores(fname))
    datasets_roc.append((scores, y_full, label, color))


# Plot parameters
static_cut_target = 0.92
rej_target        = 0.99

# Initialize side-by-side subplots
n_panels = len(datasets_roc)
fig_w = max(22, 14 + 1.5 * (n_panels - 4))   # wider if many panels
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(fig_w, 10))
fig.suptitle(f"ROC Curve Analysis — Static vs Adaptive Cut  "
             f"(scenario: {SCENARIO})",
             fontsize=22, fontweight='bold', y=0.96)

ax_ins1 = inset_axes(ax1, width="40%", height="40%", loc='upper right',
                     bbox_to_anchor=(-0.02, -0.02, 1, 1),
                     bbox_transform=ax1.transAxes)
ax_ins2 = inset_axes(ax2, width="40%", height="40%", loc='upper right',
                     bbox_to_anchor=(-0.02, -0.02, 1, 1),
                     bbox_transform=ax2.transAxes)

# Loop over datasets

for scores, labels, name, color in datasets_roc:
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    auc_score = auc(fpr, tpr)

    # ---- LEFT PANEL: STATIC CUT ----
    idx_static = np.where(thresholds >= static_cut_target)[0]
    idx_s = idx_static[-1] if len(idx_static) > 0 else 0
    sig_eff_s = float(tpr[idx_s])
    bkg_rej_s = float(1 - fpr[idx_s])

    ax1.plot(tpr, 1 - fpr, color=color, linewidth=2,
             label=f"{name:<14} | AUC={auc_score:.4f} | "
                   f"Eff={sig_eff_s*100:5.1f}% | "
                   f"Rej={bkg_rej_s*100:5.2f}%")
    ax1.plot(tpr[idx_s], 1 - fpr[idx_s], 'o', color=color, markersize=8)
    ax_ins1.plot(tpr, 1 - fpr, color=color, linewidth=1.5)
    ax_ins1.plot(tpr[idx_s], 1 - fpr[idx_s], 'o', color=color, markersize=6)

    # ---- RIGHT PANEL: ADAPTIVE CUT ----
    idx_dyn = np.where(1 - fpr >= rej_target)[0]
    idx_d = idx_dyn[-1] if len(idx_dyn) > 0 else 0
    sig_eff_d = float(tpr[idx_d])
    opt_cut   = float(thresholds[idx_d])

    ax2.plot(tpr, 1 - fpr, color=color, linewidth=2,
             label=f"{name:<14} | AUC={auc_score:.4f} | "
                   f"Cut={opt_cut:.3f} | "
                   f"Eff={sig_eff_d*100:5.1f}%")
    ax2.plot(tpr[idx_d], 1 - fpr[idx_d], 'o', color=color, markersize=8)
    ax_ins2.plot(tpr, 1 - fpr, color=color, linewidth=1.5)
    ax_ins2.plot(tpr[idx_d], 1 - fpr[idx_d], 'o', color=color, markersize=6)

# Format LEFT subplot
ax1.plot([0, 1], [1, 0], 'k--', alpha=0.3)
ax1.set_xlabel("Signal Efficiency (TPR)", fontsize=12, fontweight='bold')
ax1.set_ylabel("Background Rejection (1 − FPR)", fontsize=12, fontweight='bold')
ax1.set_title(f"A. Operating Points at Static BDT Cut = {static_cut_target:.2f}",
              fontsize=13, fontweight='bold', color='crimson')
ax1.grid(alpha=0.3)
ax1.legend(loc='lower left', fontsize=11, framealpha=0.9)

ax_ins1.set_xlim(0.35, 0.82)
ax_ins1.set_ylim(0.915, 1.002)
ax_ins1.set_title("Operating-point scatter (zoom)", fontsize=9, fontweight='bold')
ax_ins1.grid(alpha=0.3)
mark_inset(ax1, ax_ins1, loc1=2, loc2=3, fc="none", ec="0.5", ls="--")

# Format RIGHT subplot

ax2.plot([0, 1], [1, 0], 'k--', alpha=0.3)
ax2.set_xlabel("Signal Efficiency (TPR)", fontsize=12, fontweight='bold')
ax2.set_ylabel("Background Rejection (1 − FPR)", fontsize=12, fontweight='bold')
ax2.set_title(f"B. Operating Points at Adaptive Cut (Rej = {rej_target*100:.0f}%)",
              fontsize=13, fontweight='bold', color='darkgreen')
ax2.grid(alpha=0.3)
ax2.legend(loc='lower left', fontsize=11, framealpha=0.9)

ax_ins2.set_xlim(0.50, 0.62)
ax_ins2.set_ylim(0.988, 1.001)
ax_ins2.set_title("Stabilized 99% rejection line (zoom)", fontsize=9, fontweight='bold')
ax_ins2.grid(alpha=0.3)
mark_inset(ax2, ax_ins2, loc1=2, loc2=3, fc="none", ec="0.5", ls="--")

plt.tight_layout(rect=[0, 0, 1, 0.94])
# plt.savefig(f"roc_robustness_{SCENARIO}.png", dpi=200, bbox_inches='tight")
plt.show()


# KS Test 
https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.kstest.html

Performs the (one-sample or two-sample) Kolmogorov-Smirnov test for goodness of fit.

The one-sample test compares the underlying distribution F(x) of a sample against a given distribution G(x). The two-sample test compares the underlying distributions of two independent samples. Both tests are valid only for continuous distributions.

In [ ]:
from scipy.stats import ks_2samp
import pandas as pd  

In [ ]:
# TABLE HEADER & SEPARATOR
print(f"{'Dataset':<20} {'AUC':>8} {'Sig eff':>9} {'BDT cut':>9} "
      f"{'Mean shift':>13} {'% sig flipped':>15} {'Δsig':>7} {'Δbkg':>7} {'KS (momerr)':>16}")
print("-" * 108)

# Baseline row (must match loop format exactly for alignment)
print(f"{'Baseline (orig)':<20} {auc(fpr0,tpr0):>8.4f} {tpr0[cut_idx]*100:>7.2f}% "
      f"{bdt_cut:>9.4f} {' 0.000000':>13} {'    0.00%':>15} {0:>7} {0:>7} {'N/A':>16}")

# LOOP OVER SCENARIOS
SCENARIO = "momerr"

for label, fname in files_for(SCENARIO):
    x_new = get_features_from_file(fname)

    # Extract momerr for KS test (handles both DataFrame and numpy array outputs)
    if isinstance(x_new, pd.DataFrame):
        new_momerr = x_new["momerr"].values
    else:
        new_momerr = x_new[:, momerr_idx]

    # Compute KS only when momerr is perturbed
    if "momerr" in label:
        ks_stat, p_val = ks_2samp(orig_momerr, new_momerr)
        ks_str = f"{ks_stat:.4f}"
    else:
        ks_str = "N/A"

    # Rest of your evaluation pipeline (unchanged)
    y_new = model.predict_proba(x_new)[:, 1]
    delta = y_new - y_pred_original_full
    fpr, tpr, thr = roc_curve(y_full, y_new, pos_label=1)
    auc_score     = auc(fpr, tpr)

    idx        = np.where(1 - fpr >= 0.99)[0]
    cut_i      = idx[-1] if len(idx) > 0 else 0
    sig_eff    = tpr[cut_i]
    opt_cut    = thr[cut_i]

    pass_orig  = y_pred_original_full >= bdt_cut
    pass_new   = y_new                >= bdt_cut
    flipped    = (pass_orig != pass_new)
    sig_flipped = (flipped & (y_full == 1)).sum()
    bkg_flipped = (flipped & (y_full == 0)).sum()
    pct_sig     = sig_flipped / N_signal * 100.0

    # Print row with identical column widths as baseline for perfect alignment
    print(f"{label:<20} {auc_score:>8.4f} {sig_eff*100:>7.2f}% "
          f"{opt_cut:>9.4f} {delta.mean():>+13.6f} {pct_sig:>15.2f}% "
          f"{sig_flipped:>7} {bkg_flipped:>7} {ks_str:>16}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

# 1) Prepare data (ensure we only use valid numbers)
data_orig = orig_momerr[~np.isnan(orig_momerr)]
data_pert = new_momerr[~np.isnan(new_momerr)]

# 2) Perform the KS Test
ks_stat, p_val = ks_2samp(data_orig, data_pert)

# 3) Calculate CDF points manually for plotting (Robust against SciPy versions)
# We sort the data first so the line goes from bottom-left to top-right
x_orig = np.sort(data_orig)
y_orig = np.arange(1, len(x_orig) + 1) / len(x_orig) # CDF steps

x_pert = np.sort(data_pert)
y_pert = np.arange(1, len(x_pert) + 1) / len(x_pert) # CDF steps

# 4) Create the figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4), gridspec_kw={'width_ratios': [1.5, 1]})

# LEFT: Histogram Comparison (Visual)
bins = np.linspace(min(data_orig.min(), data_pert.min()), 
                   max(data_orig.max(), data_pert.max()), 60)
ax1.hist(data_orig, bins=bins, density=True, alpha=0.6, label='Original', color="#E72020")
ax1.hist(data_pert, bins=bins, density=True, alpha=0.6, label='Perturbed (×2)', color="#1105F4")
ax1.set_xlabel('momerr', fontsize=13)
ax1.set_ylabel('Probability Density', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)


zoom_min = np.percentile(data_orig, 0.5)
zoom_max = np.percentile(data_pert, 99.5)

ax1.set_xlim(zoom_min - 0.01, zoom_max + 0.01)
ax2.set_xlim(zoom_min - 0.01, zoom_max + 0.01)
# RIGHT: ECDF Overlay + KS Annotation (Statistical Truth)
ax2.plot(x_orig, y_orig, 'b', lw=2, label='Original CDF') #Comulative distrubtion function
ax2.plot(x_pert, y_pert, 'r--', lw=2, label='Perturbed CDF')

# Add the 0.5 line for reference (median)
ax2.axhline(0.5, c='gray', ls=':', alpha=0.5)

ax2.set_xlabel('momerr', fontsize=13)
ax2.set_ylabel('Cumulative Probability', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)

# Annotate KS result directly on the plot
ax2.text(0.5, 0.5, f'KS Statistic: {ks_stat:.4f}\nP-value: {p_val:.2e}', 
         transform=ax2.transAxes, va='center', ha='center', fontsize=12, 
         fontweight='bold', bbox=dict(boxstyle='round,pad=0.5', facecolor='wheat', alpha=0.85))

plt.tight_layout()
plt.show()

print(f"KS Statistic (Effect Size): {ks_stat:.4f}")